# 第8章 グラフィカルモデル：演習問題 (Exercises 8.1 - 8.29)

本ノートブックでは、PRML第8章「グラフィカルモデル (Graphical Models)」の**全29問 (Exercises 8.1 〜 8.29)** に対する詳細な数理解説、証明ステップ、穴埋め問題、および Python による数値検証コードを完全網羅しています。

### 主な学習項目
1. **ベイジアンネットワークと有向グラフ (8.1 - 8.11)**:
   - DAG結合分布の規格化性 (8.1)
   - 有向非巡回性 (DAG) とトポロジカルソートの同値性 (8.2)
   - Table 8.2 条件付き独立性と周辺依存性 (8.3, 8.4)
   - 関連度ベクトルマシン (RVM) のプレート有向グラフィカルモデル (8.5)
   - Noisy-OR 条件付き確率分布とパラメータ削減 (8.6)
   - 線形ガウスモデルの平均・分散再帰関係式 (8.7)
   - 半グラフ性分解公理 $a \perp\!\!\!\perp b, c | d \implies a \perp\!\!\!\perp b | d$ (8.8)
   - 有向グラフにおけるマルコフブランケットとd分離 (8.9)
   - 子孫観測による head-to-head 活性化 (8.10)
   - 車の燃料系・不確実な報告モデルと相殺効果 (Explaining Away) (8.11)
2. **マルコフ確率場 (MRF) と条件付き独立性 (8.12 - 8.17)**:
   - $M$ 変数の相異なる無向グラフ総数 $2^{M(M-1)/2}$ (8.12)
   - ICM (Iterated Conditional Modes) 局所エネルギー差分の導出 (8.13)
   - 相互作用項ゼロ時の最尤配位自明性 (8.14)
   - 連鎖モデルにおける隣接ペア結合周辺分布の局所因数分解 (8.15)
   - 終端観測 $x_N$ 下での後向きメッセージ伝播 (8.16)
   - 連鎖モデルにおける条件付き独立性とメッセージ遮断 (8.17)
3. **因子グラフと確率伝播アルゴリズム (8.18 - 8.29)**:
   - 有向木と無向木の相互変換・根ノード選択数 (8.18)
   - 連鎖因子グラフでの Sum-Product と Forward-Backward 等価性 (8.19)
   - 木構造因子グラフにおけるメッセージ通過プロトコルの帰納法的証明 (8.20)
   - 因子周辺分布 $p(\mathbf{x}_s)$ の Sum-Product による算出 (8.21)
   - 連結部分集合に対する周辺分布計算 (8.22)
   - 単一リンク上の対向メッセージ積による変数周辺分布表現 (8.23)
   - 因子周辺公式 (8.72) の部分木分解証明 (8.24)
   - Figure 8.51 因子グラフの周辺分布・結合分布の厳密導出 (8.25)
   - クランピングを用いた非隣接変数ペア結合分布 $p(x_a, x_b)$ の計算 (8.26)
   - 個別周辺最大化と結合最尤の乖離反例 $p(\widehat{x}, \widehat{y}) = 0$ (8.27)
   - 閉路グラフにおける保留中メッセージの非停止性 (8.28)
   - 木構造グラフにおけるメッセージ伝播の有限回停止性 ($2|E|$) (8.29)


In [1]:
# 共通環境のインポートと数値設定
import sys, os
sys.path.append(os.path.abspath('../'))
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

from common.graphical_models_utils import check_d_separation, denoise_image_icm, SimpleFactorGraphChain

np.random.seed(42)
print("Environment successfully initialized with numpy and graphical_models_utils.")

Environment successfully initialized with numpy and graphical_models_utils.


---
## <a id="Exercise-8.1"></a>Exercise 8.1: DAG 結合分布の規格化性

### 問題の提示
有向非巡回グラフ (DAG) の結合確率分布の表現
$$ p(\mathbf{x}) = \prod_{k=1}^K p(x_k | \mathrm{pa}_k) \quad (8.5) $$
において、各局所条件付き分布 $p(x_k | \mathrm{pa}_k)$ が規格化されているならば、末端から順に変数を周辺化することで、全体の結合分布も正しく規格化（総和が $1$）されていることを示せ。

### [解答の道筋と穴埋め]
1. **トポロジカル順序における周辺化**:
   DAG には有向閉路が存在しないため、親ノードが常に子ノードより前に現れるトポロジカル順序 $x_1, x_2, \dots, x_K$ が存在する。
   全体の総和（または積分）をこの逆順（葉ノード $x_K$ から根ノード $x_1$）に行う：
   $$ \sum_{x_1} \dots \sum_{x_K} \prod_{k=1}^K p(x_k | \mathrm{pa}_k) = \sum_{x_1} \dots \sum_{x_{K-1}} \left( \prod_{k=1}^{K-1} p(x_k | \mathrm{pa}_k) \right) \left[ \sum_{x_K} [ \text{①} ] \right] $$
2. **局所規格化条件の再帰的適用**:
   ノード $x_K$ は葉ノードであるため、他のいかなるノード $x_1, \dots, x_{K-1}$ の親にもなり得ない。したがって最後の因数のみが $x_K$ に依存し、
   $$ \sum_{x_K} p(x_K | \mathrm{pa}_K) = 1 $$
   となる。次に同様に $x_{K-1}$ について周辺化すると：
   $$ \sum_{x_{K-1}} p(x_{K-1} | \mathrm{pa}_{K-1}) = [ \text{②} ] $$
   これを帰納的に繰り返すことで、最終的に $\sum_{x_1} p(x_1) = 1$ となり、全体が厳密に $1$ に規格化される。

### 穴埋めの解答
- ①: $p(x_K | \mathrm{pa}_K)$
- ②: $1$

In [2]:
# Exercise 8.1 数値検証: 4ノードDAGにおける結合分布の完全規格化と順次周辺化
# グラフ: x1 -> x2 -> x4, x1 -> x3 -> x4 (ダイヤモンドDAG)
# 各変数は二値 {0, 1}
p_x1 = np.array([0.6, 0.4]) # p(x1)
p_x2_given_x1 = np.array([[0.8, 0.2], [0.3, 0.7]]) # p(x2 | x1)
p_x3_given_x1 = np.array([[0.5, 0.5], [0.1, 0.9]]) # p(x3 | x1)
# p(x4 | x2, x3): shape (2, 2, 2)
p_x4_given_x2x3 = np.zeros((2, 2, 2))
p_x4_given_x2x3[0, 0] = [0.9, 0.1]
p_x4_given_x2x3[0, 1] = [0.6, 0.4]
p_x4_given_x2x3[1, 0] = [0.4, 0.6]
p_x4_given_x2x3[1, 1] = [0.2, 0.8]

# 結合分布テンソルの構築 p(x1, x2, x3, x4)
joint = np.zeros((2, 2, 2, 2))
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            for x4 in range(2):
                joint[x1, x2, x3, x4] = (p_x1[x1] * 
                                         p_x2_given_x1[x1, x2] * 
                                         p_x3_given_x1[x1, x3] * 
                                         p_x4_given_x2x3[x2, x3, x4])

# 1. 全結合確率の総和が 1 であること
total_sum = np.sum(joint)
np.testing.assert_allclose(total_sum, 1.0, atol=1e-12)

# 2. 葉ノード x4 を周辺化したテンソルが p(x1)p(x2|x1)p(x3|x1) と厳密一致すること
marg_x4 = np.sum(joint, axis=3) # sum over x4
expected_123 = np.zeros((2, 2, 2))
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            expected_123[x1, x2, x3] = p_x1[x1] * p_x2_given_x1[x1, x2] * p_x3_given_x1[x1, x3]

np.testing.assert_allclose(marg_x4, expected_123, atol=1e-12)
print(f"Exercise 8.1 verified: Total DAG joint sum is {total_sum:.12f}, and sequential marginalization holds strictly.")

Exercise 8.1 verified: Total DAG joint sum is 1.000000000000, and sequential marginalization holds strictly.


---
## <a id="Exercise-8.2"></a>Exercise 8.2: 有向非巡回性 (DAG) とトポロジカルソートの同値性

### 問題の提示
有向グラフにおいて「有向閉路（directed cycle）が存在しないこと」と「すべての有向リンクが番号の小さいノードから大きいノードへ向かうようにノードに順序番号を付与できること（トポロジカルソートの存在）」が同値であることを示せ。

### [解答の道筋と穴埋め]
1. **十分条件（順序付けが存在する $\implies$ 閉路なし）**:
   ノード集合 $V$ の各要素 $v$ に番号 $\pi(v) \in \{1, \dots, K\}$ が割り当てられ、すべての有向辺 $u \to v$ について $\pi(u) < \pi(v)$ が成り立つとする。
   いま、背理法として有向閉路 $v_1 \to v_2 \to \dots \to v_m \to v_1$ が存在すると仮定する。
   各辺の順序関係から：
   $$ \pi(v_1) < \pi(v_2) < \dots < \pi(v_m) < [ \text{①} ] $$
   となり、$\pi(v_1) < \pi(v_1)$ という矛盾が生じる。したがって有向閉路は存在し得ない。
2. **必要条件（閉路なし $\implies$ 順序付けが存在する）**:
   閉路のない有限有向グラフには、入次数（in-degree）が $0$ のノードが少なくとも1つ存在する（もし全ノードの入次数 $\ge 1$ ならば、親を遡ることで有限グラフゆえ必ず閉路が生じるため）。
   入次数 $0$ のノードに最小番号 $1$ を付与し、そのノードと接続する出力辺を取り除く。残された部分グラフも依然として閉路を持たないため、再び入次数 $0$ のノードを見つけて番号 $2$ を付与する。
   これを繰り返す（[ \text{②} ] のアルゴリズム）ことで、すべての辺が小さい番号から大きい番号へ向かう順序付けが完成する。

### 穴埋めの解答
- ①: $\pi(v_1)$
- ②: Kahn（カーン）

In [3]:
# Exercise 8.2 数値検証: Kahnのアルゴリズムによるトポロジカルソートと閉路検出
def topological_sort(num_nodes, edges):
    in_degree = [0] * num_nodes
    adj = [[] for _ in range(num_nodes)]
    for u, v in edges:
        adj[u].append(v)
        in_degree[v] += 1
        
    queue = [i for i in range(num_nodes) if in_degree[i] == 0]
    order = []
    
    while queue:
        curr = queue.pop(0)
        order.append(curr)
        for nxt in adj[curr]:
            in_degree[nxt] -= 1
            if in_degree[nxt] == 0:
                queue.append(nxt)
                
    if len(order) < num_nodes:
        raise ValueError("Directed cycle detected!")
    return order

# 1. 閉路のない有向グラフのトポロジカルソート検証
edges_dag = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (2, 4)]
order = topological_sort(5, edges_dag)
# すべての有向辺 u -> v について order.index(u) < order.index(v) を確認
pos = {node: idx for idx, node in enumerate(order)}
for u, v in edges_dag:
    assert pos[u] < pos[v], f"Edge {u}->{v} violates topological ordering!"

# 2. 閉路を追加したときの例外送出確認
edges_cycle = edges_dag + [(4, 1)] # 1 -> 3 -> 4 -> 1 の閉路
cycle_detected = False
try:
    topological_sort(5, edges_cycle)
except ValueError:
    cycle_detected = True

assert cycle_detected, "Cycle detection failed!"
print(f"Exercise 8.2 verified: Topological order {order} strictly satisfies u < v for all edges; cycle correctly detected.")

Exercise 8.2 verified: Topological order [0, 1, 2, 3, 4] strictly satisfies u < v for all edges; cycle correctly detected.


---
## <a id="Exercise-8.3"></a>Exercise 8.3: Table 8.2 条件付き独立性と周辺依存性の直接検証

### 問題の提示
Table 8.2 に与えられた3つの二値変数 $a, b, c \in \{0, 1\}$ の結合確率分布に対し、
$$ p(a, b) \neq p(a)p(b) \quad (\text{周辺的には従属}) $$
であるが、変数 $c$ で条件付けると $c = 0, c = 1$ の双方について
$$ p(a, b | c) = p(a | c)p(b | c) \quad (\text{条件付き独立}) $$
となることを直接計算により示せ。

### [解答の道筋と穴埋め]
1. **周辺確率の計算**:
   - $p(a=1) = \sum_{b,c} p(1, b, c) = 0.192 + 0.064 + 0.048 + 0.096 = 0.400$
   - $p(b=1) = \sum_{a,c} p(a, 1, c) = 0.048 + 0.216 + 0.048 + 0.096 = 0.408$
   - $p(a=1, b=1) = \sum_c p(1, 1, c) = 0.048 + 0.096 = 0.144$
   $p(a=1)p(b=1) = 0.400 \times 0.408 = [ \text{①} ]$ であるため、$p(a,b) \neq p(a)p(b)$（従属）。
2. **条件付き確率の計算**:
   - $p(c=0) = 0.192 + 0.048 + 0.192 + 0.048 = 0.480$
   - $p(c=1) = 0.144 + 0.216 + 0.064 + 0.096 = 0.520$
   $c=0$ の下で：
   $p(a=1|c=0) = \frac{0.192 + 0.048}{0.480} = 0.500$, $p(b=1|c=0) = \frac{0.048 + 0.048}{0.480} = 0.200$
   積は $0.500 \times 0.200 = 0.100$。
   一方 $p(a=1, b=1 | c=0) = \frac{0.048}{0.480} = [ \text{②} ]$。両者は完全に一致する。

### 穴埋めの解答
- ①: $0.1632$
- ②: $0.100$

In [4]:
# Exercise 8.3 数値検証: Table 8.2 の周辺依存性と条件付き独立性の完全検証
table_8_2 = {
    (0, 0, 0): 0.192, (0, 0, 1): 0.144,
    (0, 1, 0): 0.048, (0, 1, 1): 0.216,
    (1, 0, 0): 0.192, (1, 0, 1): 0.064,
    (1, 1, 0): 0.048, (1, 1, 1): 0.096,
}

# 1. 周辺依存性の検証
p_a1 = sum(v for (a, b, c), v in table_8_2.items() if a == 1)
p_b1 = sum(v for (a, b, c), v in table_8_2.items() if b == 1)
p_a1_b1 = sum(v for (a, b, c), v in table_8_2.items() if a == 1 and b == 1)

np.testing.assert_allclose(p_a1, 0.400, atol=1e-10)
np.testing.assert_allclose(p_b1, 0.408, atol=1e-10)
np.testing.assert_allclose(p_a1_b1, 0.144, atol=1e-10)
assert not np.isclose(p_a1 * p_b1, p_a1_b1), "a and b should be dependent!"

# 2. c=0 および c=1 における条件付き独立性の検証
for c_val in [0, 1]:
    p_c = sum(v for (a, b, c), v in table_8_2.items() if c == c_val)
    p_a1_given_c = sum(v for (a, b, c), v in table_8_2.items() if a == 1 and c == c_val) / p_c
    p_b1_given_c = sum(v for (a, b, c), v in table_8_2.items() if b == 1 and c == c_val) / p_c
    p_ab_given_c = sum(v for (a, b, c), v in table_8_2.items() if a == 1 and b == 1 and c == c_val) / p_c
    np.testing.assert_allclose(p_ab_given_c, p_a1_given_c * p_b1_given_c, atol=1e-10)

print("Exercise 8.3 verified: p(a,b) != p(a)p(b), but p(a,b|c) == p(a|c)p(b|c) holds for both c=0 and c=1.")

Exercise 8.3 verified: p(a,b) != p(a)p(b), but p(a,b|c) == p(a|c)p(b|c) holds for both c=0 and c=1.


---
## <a id="Exercise-8.4"></a>Exercise 8.4: Table 8.2 の因数分解 $p(a, b, c) = p(a)p(c|a)p(b|c)$ と対応する有向グラフ

### 問題の提示
Table 8.2 の結合分布から各条件付き分布 $p(a)$、$p(c|a)$、$p(b|c)$ を求め、それらの積が元の結合分布と厳密に一致することを示せ。また、この因数分解に対応する有向グラフの構造を示せ。

### [解答の道筋と穴埋め]
1. **因数分解の導出**:
   乗法定理より一般に $p(a, b, c) = p(a) p(c|a) p(b|a, c)$ である。
   Exercise 8.3 より $b \perp\!\!\!\perp a \mid c$ が成り立つため、$p(b|a, c) = p(b|c)$ と簡約される。
   したがって：
   $$ p(a, b, c) = p(a) p(c|a) [ \text{①} ] $$
2. **対応する有向グラフ**:
   各ノードの親集合は $\mathrm{pa}_a = \emptyset$、$\mathrm{pa}_c = \{a\}$、$\mathrm{pa}_b = \{c\}$ であるため、有向辺は
   $$ a \rightarrow c \rightarrow b $$
   となる（ヘッド・トゥ・テイルの直鎖構造）。またベイズの定理により $p(a)p(c|a) = p(c)p(a|c)$ と書けば $a \leftarrow c \rightarrow b$（テール・トゥ・テールの分岐構造）も同一の条件付き独立性を表す。

### 穴埋めの解答
- ①: $p(b|c)$
- ②: $a \rightarrow c \rightarrow b$

In [5]:
# Exercise 8.4 数値検証: p(a) * p(c|a) * p(b|c) による Table 8.2 の完全再構成
# p(a)
p_a = np.array([sum(v for (a, b, c), v in table_8_2.items() if a == 0),
                sum(v for (a, b, c), v in table_8_2.items() if a == 1)])

# p(c|a)
p_c_given_a = np.zeros((2, 2))
for a in [0, 1]:
    for c in [0, 1]:
        p_c_given_a[a, c] = sum(v for (x_a, b, x_c), v in table_8_2.items() if x_a == a and x_c == c) / p_a[a]

# p(b|c)
p_c = np.array([sum(v for (a, b, c), v in table_8_2.items() if c == 0),
                sum(v for (a, b, c), v in table_8_2.items() if c == 1)])
p_b_given_c = np.zeros((2, 2))
for c in [0, 1]:
    for b in [0, 1]:
        p_b_given_c[c, b] = sum(v for (a, x_b, x_c), v in table_8_2.items() if x_c == c and x_b == b) / p_c[c]

# 全結合配置での積の比較
for (a, b, c), true_val in table_8_2.items():
    recon = p_a[a] * p_c_given_a[a, c] * p_b_given_c[c, b]
    np.testing.assert_allclose(recon, true_val, atol=1e-10)

print("Exercise 8.4 verified: p(a,b,c) == p(a)p(c|a)p(b|c) holds across all 8 states to machine precision.")

Exercise 8.4 verified: p(a,b,c) == p(a)p(c|a)p(b|c) holds across all 8 states to machine precision.


---
## <a id="Exercise-8.5"></a>Exercise 8.5: 関連度ベクトルマシン (RVM) の有向グラフィカルモデル

### 問題の提示
PRML式 (7.79) および (7.80) で記述される関連度ベクトルマシン（RVM）に対する有向確率的グラフィカルモデルをプレート記法を用いて描け。

### [解答の道筋と穴埋め]
1. **階層構造と変数間の因数分解**:
   - 各重み $w_i$ ($i = 1, \dots, M$) は独立なハイパーパラメータ $\alpha_i$ を精度とする事前ガウス分布に従う：
     $$ p(\mathbf{w}|\boldsymbol{\alpha}) = \prod_{i=1}^M [ \text{①} ] $$
   - 観測目標値 $t_n$ ($n = 1, \dots, N$) は、入力 $\mathbf{x}_n$、重み $\mathbf{w}$、ノイズ精度 $\beta$ が与えられたとき条件付き独立である：
     $$ p(\mathbf{t}|\mathbf{w}, \mathbf{X}, \beta) = \prod_{n=1}^N \mathcal{N}(t_n | \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_n), \beta^{-1}) $$
2. **プレート記法**:
   - ノード $\alpha_i$ から $w_i$ へのリンクが存在し、これが $M$ 個並ぶプレートを構成する。
   - $w_i$ からは $N$ 個の観測ノード $t_n$ へ矢印が向かい、観測入力 $\mathbf{x}_n$ および精度 $\beta$ も $t_n$ に向かう。
   - 重み $\mathbf{w}$ を条件付けると、すべての観測値 $t_n$ は互いに [ \text{②} ] となる。

### 穴埋めの解答
- ①: $\mathcal{N}(w_i | 0, \alpha_i^{-1})$
- ②: 条件付き独立

In [6]:
# Exercise 8.5 数値検証: RVM グラフの因数分解と d-分離判定
# ノード: alpha, w, beta, x1, t1, x2, t2
# 有向辺: alpha -> w, w -> t1, w -> t2, beta -> t1, beta -> t2, x1 -> t1, x2 -> t2
rvm_adj = {
    'alpha': ['w'],
    'beta': ['t1', 't2'],
    'w': ['t1', 't2'],
    'x1': ['t1'],
    'x2': ['t2'],
    't1': [],
    't2': []
}

# 1. w を条件付けたとき、t1 と t2 は条件付き独立か (d-分離判定)
d_sep_t1_t2 = check_d_separation(rvm_adj, ['t1'], ['t2'], ['w', 'beta'])
assert d_sep_t1_t2, "t1 and t2 must be d-separated given w and beta"

# 2. w を観測しないとき、t1 と t2 は従属
d_sep_uncond = check_d_separation(rvm_adj, ['t1'], ['t2'], [])
assert not d_sep_uncond, "t1 and t2 are marginally dependent through latent w"

print(f"Exercise 8.5 verified: RVM graph satisfies (t1 _|_ t2 | w, beta) = {d_sep_t1_t2}, confirming plate conditional independence.")

Exercise 8.5 verified: RVM graph satisfies (t1 _|_ t2 | w, beta) = True, confirming plate conditional independence.


---
## <a id="Exercise-8.6"></a>Exercise 8.6: Noisy-OR 条件付き確率分布とパラメータ削減

### 問題の提示
二値原因変数 $x_i \in \{0, 1\}$ ($i=1, \dots, M$) と二値結果変数 $y \in \{0, 1\}$ に対する Noisy-OR 分布 (Pearl, 1988) は次式で定義される：
$$ p(y = 1 | x_1, \dots, x_M) = 1 - (1 - \mu_0) \prod_{i=1}^M (1 - \mu_i)^{x_i} \quad (8.104) $$
ここで $\mu_i = p(y=1|x_i=1, \mathbf{x}_{\setminus i}=\mathbf{0}, \mu_0=0)$ は各原因 $i$ が単独で発症させる確率であり、$\mu_0$ は背景自発確率である。
この表現が論理和（OR関数）の確率的「ソフト」拡張であることを示し、パラメータ数が $2^M$ から $M+1$ へ線形に抑制される理由を論ぜよ。

### [解答の道筋と穴埋め]
1. **確率的 OR の解釈**:
   結果が「不発（$y=0$）」に終わる確率を考える：
   $$ p(y = 0 | \mathbf{x}) = (1 - \mu_0) \prod_{i=1}^M (1 - \mu_i)^{x_i} $$
   各原因 $x_i=1$ は独立に確率 $(1 - \mu_i)$ で発症に「失敗」する。結果が $0$ になるのは、背景要因を含めたすべての原因が同時に失敗した場合のみである。
   したがって結果が $1$ になる確率は「少なくとも1つの原因が発症に成功する確率」となり、
   $$ p(y=1|\mathbf{x}) = 1 - [ \text{①} ] $$
   となる。
2. **極限 $\mu_i \to 1, \mu_0 = 0$**:
   もしすべての $\mu_i = 1$ かつ $\mu_0 = 0$ ならば、いずれかの $x_i = 1$ である限り積は $0$ となり $p(y=1) = 1$、すべての $x_i = 0$ のときのみ $p(y=1) = 0$ となり、厳密な決定論的論理和と一致する。
3. **パラメータ数**:
   一般の条件付き分布表 (CPT) では $2^M$ 個の自由パラメータが必要であるが、Noisy-OR では各原因の信頼性 $\mu_1, \dots, \mu_M$ と背景項 $\mu_0$ の合計 [ \text{②} ] 個のみで記述できる。

### 穴埋めの解答
- ①: $(1 - \mu_0) \prod_{i=1}^M (1 - \mu_i)^{x_i}$
- ②: $M + 1$

In [7]:
# Exercise 8.6 数値検証: Noisy-OR の挙動、極限における論理和一致、パラメータ削減
M = 5
mu = np.array([0.7, 0.8, 0.6, 0.9, 0.5])
mu_0 = 0.05

def noisy_or(x, mu_weights, leak):
    # p(y=1|x) = 1 - (1 - mu_0) * prod((1 - mu_i)^x_i)
    fail_prob = (1.0 - leak) * np.prod((1.0 - mu_weights)**x)
    return 1.0 - fail_prob

# 1. すべての原因が 0 のとき、背景確率 mu_0 に一致
p_zero = noisy_or(np.zeros(M), mu, mu_0)
np.testing.assert_allclose(p_zero, mu_0, atol=1e-10)

# 2. 原因が追加されるほど y=1 の確率は単調増加する
x_a = np.array([1, 0, 0, 0, 0])
x_b = np.array([1, 1, 0, 0, 0])
assert noisy_or(x_b, mu, mu_0) >= noisy_or(x_a, mu, mu_0)

# 3. 極限 mu_i -> 1, mu_0 = 0 における論理和 (Logical OR) の完全再現
mu_sharp = np.ones(M) * 0.999999
for k in range(1 << M):
    x_vec = np.array([(k >> j) & 1 for j in range(M)])
    prob = noisy_or(x_vec, mu_sharp, 0.0)
    expected_or = float(np.any(x_vec == 1))
    np.testing.assert_allclose(prob, expected_or, atol=1e-4)

print(f"Exercise 8.6 verified: Noisy-OR satisfies leak p(y=1|0)={p_zero:.4f}, monotonic property, and recovers logical OR.")

Exercise 8.6 verified: Noisy-OR satisfies leak p(y=1|0)=0.0500, monotonic property, and recovers logical OR.


---
## <a id="Exercise-8.7"></a>Exercise 8.7: 線形ガウスモデルの平均・分散再帰関係式 (8.15 - 8.18)

### 問題の提示
各変数 $x_i$ がその親変数 $\mathrm{pa}_i$ の線形結合にガウスノイズが加わったモデル
$$ p(x_i | \mathrm{pa}_i) = \mathcal{N}\left(x_i \middle| \sum_{j \in \mathrm{pa}_i} W_{ij} x_j + b_i, v_i\right) $$
において、結合分布の平均ベクトル $\mathbb{E}[\mathbf{x}]$ および共分散行列 $\boldsymbol{\Sigma}$ の要素が次の再帰式を満たすことを示せ：
$$ \mathbb{E}[x_i] = \sum_{j \in \mathrm{pa}_i} W_{ij} \mathbb{E}[x_j] + b_i \quad (8.15) $$
$$ \operatorname{cov}(x_i, x_j) = \sum_{k \in \mathrm{pa}_j} W_{jk} \operatorname{cov}(x_i, x_k) + I_{ij} v_i \quad (8.16) $$

### [解答の道筋と穴埋め]
1. **期待値の再帰式**:
   $x_i = \sum_{j \in \mathrm{pa}_i} W_{ij} x_j + b_i + \epsilon_i$ （ここで $\mathbb{E}[\epsilon_i] = 0$）と表せる。
   両辺の期待値をとると、線形性より直ちに
   $$ \mathbb{E}[x_i] = [ \text{①} ] $$
   が得られる。トポロジカル順序に従い $i=1$ から順に計算可能である。
2. **共分散の再帰式**:
   偏差を $\widetilde{x}_i = x_i - \mathbb{E}[x_i]$ とおくと、$\widetilde{x}_j = \sum_{k \in \mathrm{pa}_j} W_{jk} \widetilde{x}_k + \epsilon_j$。
   したがって $\operatorname{cov}(x_i, x_j) = \mathbb{E}[\widetilde{x}_i \widetilde{x}_j]$ は：
   $$ \mathbb{E}\left[\widetilde{x}_i \left( \sum_{k \in \mathrm{pa}_j} W_{jk} \widetilde{x}_k + \epsilon_j \right)\right] = \sum_{k \in \mathrm{pa}_j} W_{jk} \operatorname{cov}(x_i, x_k) + \mathbb{E}[\widetilde{x}_i \epsilon_j] $$
   $i \le j$ のトポロジカル順序において、$\epsilon_j$ は先行する変数 $\widetilde{x}_i$ ($i < j$) と独立であるため $\mathbb{E}[\widetilde{x}_i \epsilon_j] = 0$。$i = j$ のときは分散 $v_i$ となり、クロネッカーのデルタ $[ \text{②} ]$ が加わる。

### 穴埋めの解答
- ①: $\sum_{j \in \mathrm{pa}_i} W_{ij} \mathbb{E}[x_j] + b_i$
- ②: $I_{ij} v_i$

In [8]:
# Exercise 8.7 数値検証: 3ノード線形ガウスDAGの再帰式解 vs 行列反転解析解
# グラフ: x1 -> x2 -> x3, x1 -> x3
b = np.array([1.0, -0.5, 2.0])
v = np.array([0.5, 0.8, 0.4])
W = np.array([
    [0.0, 0.0, 0.0],
    [0.6, 0.0, 0.0], # x2 = 0.6*x1 + b2
    [0.4, -0.7, 0.0] # x3 = 0.4*x1 - 0.7*x2 + b3
])

# 1. 再帰式 (8.15) による平均の計算
mean_rec = np.zeros(3)
for i in range(3):
    mean_rec[i] = np.dot(W[i], mean_rec) + b[i]

# 2. 再帰式 (8.16) による共分散の計算
cov_rec = np.zeros((3, 3))
for j in range(3):
    for i in range(j + 1):
        c_val = sum(W[j, k] * cov_rec[i, k] for k in range(j))
        if i == j:
            c_val += v[i]
        cov_rec[i, j] = c_val
        cov_rec[j, i] = c_val

# 3. 行列解析解: x = (I - W)^{-1} (b + eps)
I_minus_W_inv = np.linalg.inv(np.eye(3) - W)
mean_exact = I_minus_W_inv @ b
cov_exact = I_minus_W_inv @ np.diag(v) @ I_minus_W_inv.T

np.testing.assert_allclose(mean_rec, mean_exact, atol=1e-12)
np.testing.assert_allclose(cov_rec, cov_exact, atol=1e-12)
print("Exercise 8.7 verified: Recursive formulas (8.15)-(8.16) match closed-form Gaussian solution to 1e-12.")

Exercise 8.7 verified: Recursive formulas (8.15)-(8.16) match closed-form Gaussian solution to 1e-12.


---
## <a id="Exercise-8.8"></a>Exercise 8.8: 半グラフ性分解公理 $a \perp\!\!\!\perp b, c \mid d \implies a \perp\!\!\!\perp b \mid d$

### 問題の提示
確率変数の条件付き独立性に関する分解公理（decomposition property）：
$$ a \perp\!\!\!\perp b, c \mid d \implies a \perp\!\!\!\perp b \mid d $$
が成り立つことを確率の基本則から証明せよ。

### [解答の道筋と穴埋め]
1. **仮定の定式化**:
   $a \perp\!\!\!\perp b, c \mid d$ とは、条件付き同時確率が因数分解することを意味する：
   $$ p(a, b, c | d) = p(a | d) p(b, c | d) $$
2. **変数 $c$ の周辺化**:
   両辺を変数 $c$ について和（または積分）をとる：
   $$ \sum_c p(a, b, c | d) = \sum_c [ \text{①} ] p(b, c | d) $$
   $p(a|d)$ は $c$ に依存しないため和の外にくくり出すことができる：
   $$ p(a, b | d) = p(a | d) \left[ \sum_c p(b, c | d) \right] = p(a | d) [ \text{②} ] $$
   これにより $p(a, b | d) = p(a | d) p(b | d)$ が示され、命題 $a \perp\!\!\!\perp b \mid d$ が証明された。

### 穴埋めの解答
- ①: $p(a | d)$
- ②: $p(b | d)$

In [9]:
# Exercise 8.8 数値検証: a _|_ (b, c) | d から a _|_ b | d の周辺化による導出
# 乱数による確率分布の生成
np.random.seed(88)
# p(d), p(a|d), p(b,c|d)
p_d = np.array([0.4, 0.6])
p_a_given_d = np.random.dirichlet([1, 1], size=2) # (2, 2)
p_bc_given_d = np.random.dirichlet(np.ones(4), size=2).reshape(2, 2, 2) # (2, 2, 2) for b, c

# 結合分布 p(a, b, c, d) = p(d) * p(a|d) * p(bc|d)
joint_abcd = np.zeros((2, 2, 2, 2))
for d in range(2):
    for a in range(2):
        for b in range(2):
            for c in range(2):
                joint_abcd[a, b, c, d] = p_d[d] * p_a_given_d[d, a] * p_bc_given_d[d, b, c]

# c を周辺化して p(a, b | d) を計算
p_abd = np.sum(joint_abcd, axis=2) # sum over c
p_b_given_d = np.sum(p_bc_given_d, axis=2) # sum over c -> (2, 2)

for d in range(2):
    p_ab_d = p_abd[:, :, d] / p_d[d]
    expected_ab_d = np.outer(p_a_given_d[d], p_b_given_d[d])
    np.testing.assert_allclose(p_ab_d, expected_ab_d, atol=1e-12)

print("Exercise 8.8 verified: Marginalizing out c strictly yields p(a,b|d) == p(a|d)p(b|d).")

Exercise 8.8 verified: Marginalizing out c strictly yields p(a,b|d) == p(a|d)p(b|d).


---
## <a id="Exercise-8.9"></a>Exercise 8.9: 有向グラフにおけるマルコフブランケットと d-分離

### 問題の提示
d-分離基準を用いて、有向グラフにおけるあるノード $x_i$ の条件付き分布 $p(x_i | \mathbf{x}_{\setminus i})$ が、その**マルコフブランケット**
$$ \mathrm{MB}(x_i) = \mathrm{pa}(x_i) \cup \mathrm{ch}(x_i) \cup \mathrm{coparents}(x_i) $$
（親ノード、子ノード、およびその子ノードの他の親ノード）に含まれる変数のみに依存し、グラフ内のそれ以外のすべての変数から条件付き独立になることを示せ。

### [解答の道筋と穴埋め]
1. **グラフ上の任意パスの分類**:
   ノード $x_i$ からマルコフブランケットの外側にある任意のノード $y$ への無向パスを考える。このパスが $x_i$ を出発するリンクには2通りの場合がある：
   - **$x_i$ から親 $p \in \mathrm{pa}(x_i)$ へ向かうパス**:
     親 $p$ は $\mathrm{MB}(x_i)$ に含まれ観測されている。$x_i \leftarrow p$ のパスにおいて $p$ は head-to-head ではないため、観測された $p$ によってパスは [ \text{①} ] される。
   - **$x_i$ から子 $c \in \mathrm{ch}(x_i)$ へ向かうパス**:
     子 $c$ も $\mathrm{MB}(x_i)$ に含まれる。パスが $c$ からさらにその子へ向かう（$x_i \to c \to \dots$）場合、$c$ は non-head-to-head で観測されているためブロックされる。
     パスが $c$ から別の親 $cp$（$x_i$ の共親）へ向かう（$x_i \to c \leftarrow cp$）場合、$c$ は head-to-head であるが、共親 $cp$ 自身が $\mathrm{MB}(x_i)$ に含まれ観測されているため、この直後のリンクでブロックされる。
2. **結論**:
   したがって、$x_i$ から外部ノードへのすべてのパスは $\mathrm{MB}(x_i)$ により遮断されるため、d-分離の定義より
   $$ x_i \perp\!\!\!\perp (V \setminus (\{x_i\} \cup \mathrm{MB}(x_i))) \mid [ \text{②} ] $$
   が成立する。

### 穴埋めの解答
- ①: ブロック（遮断）
- ②: $\mathrm{MB}(x_i)$

In [10]:
# Exercise 8.9 数値検証: 複雑なDAGにおける全ノードのマルコフブランケットd-分離判定
# グラフ: 0 -> 1 -> 3, 2 -> 1, 1 -> 4, 5 -> 4
# ノード 1 の親: {0, 2}, 子: {3, 4}, 共親 (子4の親): {5}
dag_test = {
    0: [1], 2: [1],
    1: [3, 4],
    5: [4],
    3: [], 4: []
}

mb_1 = [0, 2, 3, 4, 5]
# ノード 1 とその他のノードの分離を判定
# ノード 1 からそれ以外の外部ノードが存在するグラフを作成（ノード 6 を追加: 0 <- 6, 3 -> 7）
dag_expanded = {
    6: [0], 0: [1], 2: [1],
    1: [3, 4],
    5: [4],
    3: [7], 4: [], 7: []
}
# ノード 1 のマルコフブランケットは依然として {0, 2, 3, 4, 5}
# ノード 6 (祖先外) および ノード 7 (子孫外) との d-分離を検証
d_sep_6 = check_d_separation(dag_expanded, [1], [6], mb_1)
d_sep_7 = check_d_separation(dag_expanded, [1], [7], mb_1)

assert d_sep_6, "Node 1 must be d-separated from node 6 given MB(1)"
assert d_sep_7, "Node 1 must be d-separated from node 7 given MB(1)"
print("Exercise 8.9 verified: Node 1 is strictly d-separated from all remaining variables given its Markov blanket.")

Exercise 8.9 verified: Node 1 is strictly d-separated from all remaining variables given its Markov blanket.


---
## <a id="Exercise-8.10"></a>Exercise 8.10: 子孫観測による head-to-head (v-構造) 活性化 (Figure 8.54)

### 問題の提示
Figure 8.54 に示された有向グラフ $a \rightarrow c \leftarrow b$、$c \rightarrow d$ において、
1. いかなる変数も観測されていないとき $a \perp\!\!\!\perp b \mid \emptyset$ となることを示せ。
2. ノード $c$ の子孫であるノード $d$ を観測したとき、一般に $a \not\perp\!\!\!\perp b \mid d$ となることを示せ。

### [解答の道筋と穴埋め]
1. **条件付けなしの場合**:
   パス $a \to c \leftarrow b$ においてノード $c$ は head-to-head（合流型）結合である。
   $c$ およびその子孫（$d$）のいずれも観測集合 $\emptyset$ に含まれていないため、このパスは [ \text{①} ] されている。
   したがって $a \perp\!\!\!\perp b \mid \emptyset$（無条件で周辺独立）。
2. **ノード $d$ を観測した場合**:
   ノード $d$ は合流ノード $c$ の子孫である。d-分離の定義により、head-to-head ノード自身またはその子孫が観測集合に含まれると、その合流ノードは活性化（非ブロック化）される。
   したがってパス $a \to c \leftarrow b$ が [ \text{②} ] となり、一般に $a$ と $b$ は条件付き従属となる。

### 穴埋めの解答
- ①: ブロック（遮断）
- ②: アクティブ（非ブロック）

In [11]:
# Exercise 8.10 数値検証: 子孫観測による head-to-head 結合の活性化
fig8_54 = {
    'a': ['c'],
    'b': ['c'],
    'c': ['d'],
    'd': []
}

# 1. d-分離判定
d_sep_empty = check_d_separation(fig8_54, ['a'], ['b'], [])
d_sep_given_d = check_d_separation(fig8_54, ['a'], ['b'], ['d'])

assert d_sep_empty, "a and b must be d-separated given empty set"
assert not d_sep_given_d, "a and b must NOT be d-separated given descendant d"

# 2. 具体的な確率分布での従属性の数値確認
# a, b ~ Bernoulli(0.5) 独立
# c = a XOR b (二値加算)
# d = c with noise (p(d=c) = 0.9)
p_a = np.array([0.5, 0.5])
p_b = np.array([0.5, 0.5])
joint = np.zeros((2, 2, 2)) # a, b, d
for a in range(2):
    for b in range(2):
        c = a ^ b
        for d in range(2):
            p_d_c = 0.9 if d == c else 0.1
            joint[a, b, d] = p_a[a] * p_b[b] * p_d_c

# d=0 を条件付けたときの p(a, b | d=0)
p_d0 = np.sum(joint[:, :, 0])
p_ab_given_d0 = joint[:, :, 0] / p_d0
p_a_given_d0 = np.sum(p_ab_given_d0, axis=1)
p_b_given_d0 = np.sum(p_ab_given_d0, axis=0)

# p(a, b|d=0) != p(a|d=0) p(b|d=0) の確認
assert not np.allclose(p_ab_given_d0, np.outer(p_a_given_d0, p_b_given_d0))
print(f"Exercise 8.10 verified: a _|_ b | empty is {d_sep_empty}, but a _|_ b | d is {d_sep_given_d}!")

Exercise 8.10 verified: a _|_ b | empty is True, but a _|_ b | d is False!


---
## <a id="Exercise-8.11"></a>Exercise 8.11: 燃料計と運転手報告モデルにおける相殺効果 (Explaining Away)

### 問題の提示
Figure 8.21 の燃料計問題において、燃料計 $G$ を直接見る代わりに、運転手 $D$ が「満タン $D=1$」か「空 $D=0$」かを報告するとする。
運転手はやや信頼性に欠け、次の条件付き確率で報告を行う：
$$ p(D=1|G=1) = 0.9, \quad p(D=0|G=0) = 0.9 $$
運転手が「空 $D=0$」と報告したとき、
1. この報告のみが与えられたときの燃料タンクが空である確率 $p(F=0|D=0)$ を求めよ。
2. さらにバッテリーが上がっていること（$B=0$）を観察したときの確率 $p(F=0|D=0, B=0)$ を求め、後者の確率が低下することを示せ。
3. この結果の直観的意味（Explaining Away）および Figure 8.54 との関係を論ぜよ。

### [解答の道筋と穴埋め]
1. **事前確率と燃料計 CPT**:
   PRML本文より、$p(B=1)=0.9$、$p(F=1)=0.9$。
   燃料計はバッテリーと燃料の双方が揃っているときのみ動き、
   $p(G=1|B=1,F=1)=0.8$、それ以外の組み合わせでは $p(G=1|B,F)=0$（故障率を無視した理想化モデル）とする。
2. **$p(F=0|D=0)$ の評価**:
   観測 $D=0$ のもとでベイズの定理を適用すると、運転手の「空」報告は燃料切れの事後確率 $p(F=0|D=0)$ を事前確率 $0.10$ から大きく上昇させる。
3. **$B=0$ が与えられたときの相殺効果**:
   バッテリーが上がっている（$B=0$）ことが分かると、燃料計が動かなかった直接の原因が「バッテリー上がり」によって完全に説明される（[ \text{①} ]）。
   その結果、「燃料切れ」である必要性が薄れ、事後確率 $p(F=0|D=0, B=0)$ は $p(F=0|D=0)$ より [ \text{②} ] する。
   これはノード $D$ を介して合流ノード $G$ の子孫が観測されたことで生じた典型的な v-構造の相殺効果である。

### 穴埋めの解答
- ①: 相殺（Explaining Away）
- ②: 低下（減少）

In [12]:
# Exercise 8.11 数値検証: 運転手報告モデルにおける相殺効果の厳密計算
p_B1, p_F1 = 0.9, 0.9
p_B = np.array([1.0 - p_B1, p_B1])
p_F = np.array([1.0 - p_F1, p_F1])

p_G1_given_BF = np.zeros((2, 2))
p_G1_given_BF[1, 1] = 0.8
p_G1_given_BF[0, 1] = 0.0
p_G1_given_BF[1, 0] = 0.0
p_G1_given_BF[0, 0] = 0.0

p_D1_given_G = np.array([0.1, 0.9]) # p(D=1|G=0)=0.1, p(D=1|G=1)=0.9

# 4変数結合分布 p(B, F, G, D)
joint_BFGD = np.zeros((2, 2, 2, 2))
for B in range(2):
    for F in range(2):
        p_G1 = p_G1_given_BF[B, F]
        p_G = [1.0 - p_G1, p_G1]
        for G in range(2):
            p_D1 = p_D1_given_G[G]
            p_D = [1.0 - p_D1, p_D1]
            for D in range(2):
                joint_BFGD[B, F, G, D] = p_B[B] * p_F[F] * p_G[G] * p_D[D]

# 1. p(F=0 | D=0)
p_D0 = np.sum(joint_BFGD[:, :, :, 0])
p_F0_D0 = np.sum(joint_BFGD[:, 0, :, 0])
prob_F0_given_D0 = p_F0_D0 / p_D0

# 2. p(F=0 | D=0, B=0)
p_D0_B0 = np.sum(joint_BFGD[0, :, :, 0])
p_F0_D0_B0 = np.sum(joint_BFGD[0, 0, :, 0])
prob_F0_given_D0_B0 = p_F0_D0_B0 / p_D0_B0

assert prob_F0_given_D0_B0 < prob_F0_given_D0, "Explaining away must reduce the probability of F=0"
print(f"Exercise 8.11 verified:")
print(f"  p(F=0 | D=0)        = {prob_F0_given_D0:.4f} (increased from prior 0.10)")
print(f"  p(F=0 | D=0, B=0)   = {prob_F0_given_D0_B0:.4f} (reduced due to explaining away)")

Exercise 8.11 verified:
  p(F=0 | D=0)        = 0.2358 (increased from prior 0.10)
  p(F=0 | D=0, B=0)   = 0.1000 (reduced due to explaining away)


---
## <a id="Exercise-8.12"></a>Exercise 8.12: $M$ 変数の相異なる無向グラフ総数 $2^{M(M-1)/2}$

### 問題の提示
$M$ 個の相異なる確率変数上に定義できる相異なる無向グラフの総数が
$$ 2^{M(M-1)/2} $$
であることを示せ。また $M=3$ の場合の全 8 通りの可能性を図示（分類）せよ。

### [解答の道筋と穴埋め]
1. **無向グラフのエッジ総数**:
   無向グラフにおいて、任意の2ノード間に自己ループや多重辺を持たない単純グラフを考える。
   $M$ 個のノードから相異なる2ノードを選ぶ組み合わせの総数は：
   $$ \binom{M}{2} = \frac{M(M-1)}{2} $$
2. **部分集合の数**:
   各ペア $\{u, v\}$ について、「辺が存在する」または「存在しない」の独立な $2$ 通りの選択肢がある。
   したがって相異なる無向グラフの総数は：
   $$ 2^{\binom{M}{2}} = [ \text{①} ] $$
3. **$M=3$ の分類**:
   $\binom{3}{2} = 3$ 本の潜在的エッジがあるため、グラフ総数は $2^3 = 8$ 個。エッジ本数により分類できる：
   - 0本: 完全に孤立した3ノード（1通り）
   - 1本: 1辺のみ存在（3通り）
   - 2本: 2辺の連鎖 $a-b-c$（3通り）
   - 3本: 完全三角形グラフ $K_3$（[ \text{②} ] 通り）

### 穴埋めの解答
- ①: $2^{M(M-1)/2}$
- ②: $1$

In [13]:
# Exercise 8.12 数値検証: M=3 における全8通りの無向グラフ列挙とエッジ数分類
import itertools

nodes = ['a', 'b', 'c']
pairs = list(itertools.combinations(nodes, 2))
num_pairs = len(pairs)
total_graphs = 2**num_pairs

graphs_by_edge_count = {0: [], 1: [], 2: [], 3: []}

for mask in range(total_graphs):
    edges = [pairs[i] for i in range(num_pairs) if (mask >> i) & 1]
    graphs_by_edge_count[len(edges)].append(edges)

assert total_graphs == 8
assert len(graphs_by_edge_count[0]) == 1
assert len(graphs_by_edge_count[1]) == 3
assert len(graphs_by_edge_count[2]) == 3
assert len(graphs_by_edge_count[3]) == 1

print(f"Exercise 8.12 verified: Exactly 2^(3*2/2) = {total_graphs} graphs:")
for k, g_list in graphs_by_edge_count.items():
    print(f"  {k} edges ({len(g_list)} graphs): {g_list}")

Exercise 8.12 verified: Exactly 2^(3*2/2) = 8 graphs:
  0 edges (1 graphs): [[]]
  1 edges (3 graphs): [[('a', 'b')], [('a', 'c')], [('b', 'c')]]
  2 edges (3 graphs): [[('a', 'b'), ('a', 'c')], [('a', 'b'), ('b', 'c')], [('a', 'c'), ('b', 'c')]]
  3 edges (1 graphs): [[('a', 'b'), ('a', 'c'), ('b', 'c')]]


---
## <a id="Exercise-8.13"></a>Exercise 8.13: ICM 局所エネルギー差分 $\Delta E(x_j)$ の導出

### 問題の提示
画像ノイズ除去モデル (PRML 8.3.3節) のエネルギー関数
$$ E(\mathbf{x}, \mathbf{y}) = h \sum_i x_i - \beta \sum_{\{i, j\}} x_i x_j - \eta \sum_i x_i y_i \quad (8.42) $$
（ここで $x_i, y_i \in \{-1, +1\}$）において、ある特定変数 $x_j$ 以外のすべての変数を固定したとき、$x_j = +1$ と $x_j = -1$ の状態間のエネルギー差分 $\Delta E(x_j) = E(x_j=+1) - E(x_j=-1)$ を導出せよ。また、これがノード $x_j$ のグラフ局所情報（マルコフブランケット）のみに依存することを示せ。

### [解答の道筋と穴埋め]
1. **エネルギー関数の $x_j$ 依存項の分離**:
   $x_j$ を含む項のみを抽出した局所エネルギー $E_j(x_j)$ は：
   $$ E_j(x_j) = h x_j - \beta x_j \sum_{i \in \mathrm{ne}(j)} x_i - \eta x_j y_j $$
   ここで $\mathrm{ne}(j)$ はノード $x_j$ の隣接近傍ノードの集合である。
2. **エネルギー差分の計算**:
   $x_j = +1$ のとき：$E_j(+1) = h - \beta \sum_{i \in \mathrm{ne}(j)} x_i - \eta y_j$
   $x_j = -1$ のとき：$E_j(-1) = -h + \beta \sum_{i \in \mathrm{ne}(j)} x_i + \eta y_j$
   差分をとると：
   $$ \Delta E(x_j) = E(x_j=+1) - E(x_j=-1) = [ \text{①} ] $$
   この式には $x_j$ の隣接ノード $\sum_{i \in \mathrm{ne}(j)} x_i$ および自身の観測値 $y_j$ 以外の変数は一切現れず、厳密に [ \text{②} ] な情報のみで決定される。

### 穴埋めの解答
- ①: $2 \left( h - \beta \sum_{i \in \mathrm{ne}(j)} x_i - \eta y_j \right)$
- ②: 局所的（マルコフブランケット）

In [14]:
# Exercise 8.13 数値検証: 全体エネルギー差分 vs 局所式 (8.42) の完全一致
h, beta, eta = 0.1, 0.7, 1.5
H, W = 4, 4
X = np.random.choice([-1.0, 1.0], size=(H, W))
Y = np.random.choice([-1.0, 1.0], size=(H, W))

def calc_total_energy(x_grid):
    diff_h = x_grid[:, :-1] * x_grid[:, 1:]
    diff_v = x_grid[:-1, :] * x_grid[1:, :]
    return h * np.sum(x_grid) - beta * (np.sum(diff_h) + np.sum(diff_v)) - eta * np.sum(x_grid * Y)

# 任意のピクセル (r, c) を選んで比較
r, c = 2, 2
X_pos = X.copy(); X_pos[r, c] = 1.0
X_neg = X.copy(); X_neg[r, c] = -1.0

delta_E_global = calc_total_energy(X_pos) - calc_total_energy(X_neg)

neighbors_sum = X[r-1, c] + X[r+1, c] + X[r, c-1] + X[r, c+1]
delta_E_local = 2.0 * (h - beta * neighbors_sum - eta * Y[r, c])

np.testing.assert_allclose(delta_E_global, delta_E_local, atol=1e-12)
print(f"Exercise 8.13 verified: Global energy delta {delta_E_global:.8f} == Local formula delta {delta_E_local:.8f}")

Exercise 8.13 verified: Global energy delta 0.00000000 == Local formula delta 0.00000000


---
## <a id="Exercise-8.14"></a>Exercise 8.14: 相互作用ゼロ時における最尤配位の自明性 ($x_i = y_i$)

### 問題の提示
式 (8.42) のエネルギー関数において、隣接相互作用パラメータ $\beta = 0$ かつ外部バイアス $h = 0$ とした場合、潜在変数の最も確率の高い（エネルギー最小の）配位が
$$ x_i = y_i \quad (\forall i) $$
で与えられることを示せ。

### [解答の道筋と穴埋め]
1. **エネルギー関数の退化**:
   $\beta = 0, h = 0$ のとき、エネルギー関数はピクセルごとに完全に独立な和に分解される：
   $$ E(\mathbf{x}, \mathbf{y}) = -\eta \sum_i x_i y_i = \sum_i [ \text{①} ] $$
2. **大域的最小化**:
   各項 $-\eta x_i y_i$ を最小化する問題を考える。$\eta > 0$ であるため、$x_i y_i = +1$ となるとき最小値 $-\eta$ を達成する。
   $x_i, y_i \in \{-1, +1\}$ であるから、$x_i y_i = +1 \iff x_i = y_i$ である。
   したがって、各ピクセルが自身の観測値と完全に一致する配位
   $$ x_i^* = [ \text{②} ] $$
   が大域的最尤配位となる。

### 穴埋めの解答
- ①: $-\eta x_i y_i$
- ②: $y_i$

In [15]:
# Exercise 8.14 数値検証: beta=0, h=0 における ICM 最適解の y への完全一致
noisy_img = np.random.choice([-1.0, 1.0], size=(8, 8))
# beta = 0.0, h = 0.0 でノイズ除去を実行
denoised_img, _ = denoise_image_icm(noisy_img, h=0.0, beta=0.0, eta=2.1, max_iter=5)

np.testing.assert_array_equal(denoised_img, noisy_img)
print("Exercise 8.14 verified: With beta=0 and h=0, the optimal state strictly recovers x_i == y_i for all pixels.")

Exercise 8.14 verified: With beta=0 and h=0, the optimal state strictly recovers x_i == y_i for all pixels.


---
## <a id="Exercise-8.15"></a>Exercise 8.15: 連鎖モデルにおける隣接ペア結合周辺分布の局所因数分解 (8.58)

### 問題の提示
Figure 8.38 の無向連鎖グラフモデルにおいて、隣接する2つのノード $x_{n-1}, x_n$ の結合周辺確率分布 $p(x_{n-1}, x_n)$ が、前向きメッセージ $\mu_\alpha$、局所ポテンシャル $\psi_{n-1, n}$、および後向きメッセージ $\mu_\beta$ を用いて
$$ p(x_{n-1}, x_n) = \frac{1}{Z} \mu_\alpha(x_{n-1}) \psi_{n-1, n}(x_{n-1}, x_n) \mu_\beta(x_n) \quad (8.58) $$
の形で表されることを示せ。

### [解答の道筋と穴埋め]
1. **結合分布の周辺化**:
   全結合分布 $p(\mathbf{x}) = \frac{1}{Z} \prod_{m=1}^{N-1} \psi_{m, m+1}(x_m, x_{m+1})$ から、$x_{n-1}$ と $x_n$ 以外のすべての変数を周辺化（和をとる）する：
   $$ p(x_{n-1}, x_n) = \frac{1}{Z} \sum_{\mathbf{x} \setminus \{x_{n-1}, x_n\}} \prod_{m=1}^{N-1} \psi_{m, m+1}(x_m, x_{m+1}) $$
2. **和の分配法則による因数分解**:
   積を $x_{n-1}$ より左側、$x_n$ より右側、および両者をつなぐポテンシャル $\psi_{n-1, n}$ の3つに分割する：
   $$ = \frac{1}{Z} \psi_{n-1, n}(x_{n-1}, x_n) \left[ \sum_{x_1} \dots \sum_{x_{n-2}} \prod_{m=1}^{n-2} \psi_{m, m+1} \right] \left[ \sum_{x_{n+1}} \dots \sum_{x_N} \prod_{m=n}^{N-1} \psi_{m, m+1} \right] $$
   第1の角括弧は前向きメッセージ $[ \text{①} ]$ の定義そのものであり、第2の角括弧は後向きメッセージ $[ \text{②} ]$ である。これにより式 (8.58) が導出される。

### 穴埋めの解答
- ①: $\mu_\alpha(x_{n-1})$
- ②: $\mu_\beta(x_n)$

In [16]:
# Exercise 8.15 数値検証: 連鎖モデルの直接全列挙結合周辺確率 vs 式 (8.58) メッセージ積
K_states = 2
N_chain = 4
# ランダムポテンシャル行列 psi
psis = [np.random.uniform(0.5, 2.0, size=(K_states, K_states)) for _ in range(N_chain - 1)]

# 1. 全状態 (2^4 = 16) の結合分布をブルートフォース計算
all_states = []
joint_table = {}
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            for x4 in range(2):
                val = psis[0][x1, x2] * psis[1][x2, x3] * psis[2][x3, x4]
                joint_table[(x1, x2, x3, x4)] = val

Z = sum(joint_table.values())
# x2, x3 (n-1=2, n=3) の真の結合周辺分布
true_p23 = np.zeros((2, 2))
for (x1, x2, x3, x4), val in joint_table.items():
    true_p23[x2, x3] += val / Z

# 2. 前向きメッセージ mu_alpha(x2) と 後向きメッセージ mu_beta(x3)
mu_alpha_2 = np.sum(psis[0], axis=0) # sum over x1: psi_12(x1, x2)
mu_beta_3 = np.sum(psis[2], axis=1)  # sum over x4: psi_34(x3, x4)

# 式 (8.58) による再構成
p23_formula = (mu_alpha_2[:, None] * psis[1] * mu_beta_3[None, :]) / Z

np.testing.assert_allclose(true_p23, p23_formula, atol=1e-12)
print("Exercise 8.15 verified: Pairwise joint p(x_{n-1}, x_n) matches message product (8.58) to machine precision.")

Exercise 8.15 verified: Pairwise joint p(x_{n-1}, x_n) matches message product (8.58) to machine precision.


---
## <a id="Exercise-8.16"></a>Exercise 8.16: 終端観測 $x_N$ 下でのメッセージ伝播アルゴリズム

### 問題の提示
Figure 8.38 の連鎖グラフにおいて、終端ノード $x_N$ のみが観測されたときの条件付き分布 $p(x_n | x_N)$ をすべての $n \in \{1, \dots, N-1\}$ について効率的に計算するメッセージ伝播アルゴリズムを示せ。どのメッセージがどのように修正されるかを論ぜよ。

### [解答の道筋と穴埋め]
1. **観測データの境界条件化**:
   観測値 $x_N = \widehat{x}_N$ は、ノード $N$ における局所エビデンス（デルタ関数）として表現される：
   $$ \mu_\beta(x_N) = I(x_N = \widehat{x}_N) = [ \text{①} ] $$
2. **後向きメッセージの伝播**:
   この修正された境界条件から開始して、後向きメッセージを逆順に伝播させる：
   $$ \mu_\beta(x_{N-1}) = \sum_{x_N} \psi_{N-1, N}(x_{N-1}, x_N) \delta(x_N, \widehat{x}_N) = \psi_{N-1, N}(x_{N-1}, \widehat{x}_N) $$
   これ以降、通常の後向き再帰式 $\mu_\beta(x_{m-1}) = \sum_{x_m} \psi_{m-1, m} \mu_\beta(x_m)$ を $m = N-1, \dots, 2$ へ適用する。
3. **前向きメッセージの不変性**:
   ノード $1$ から順方向に伝播するメッセージ $\mu_\alpha(x_n)$ は、終端ノード $x_N$ の観測に依存しないため一切修正されない。
   各ノードの事後条件付き分布は、未修正の前向きメッセージと修正された後向きメッセージの積
   $$ p(x_n | x_N = \widehat{x}_N) \propto [ \text{②} ] $$
   として $O(N)$ の計算量で一括して求められる。

### 穴埋めの解答
- ①: $\delta(x_N, \widehat{x}_N)$
- ②: $\mu_\alpha(x_n) \mu_\beta(x_n)$

In [17]:
# Exercise 8.16 数値検証: 終端観測 x_N のメッセージ伝播 vs ブルートフォース事後分布
# N=4 の連鎖モデル
N = 4
K = 2
transitions = [np.random.uniform(0.5, 2.0, size=(K, K)) for _ in range(N - 1)]
emissions = [np.ones(K) for _ in range(N)]
# 終端 x_4 = 1 を観測 (クランプ)
target_xN = 1
emissions[-1] = np.array([0.0, 1.0])

chain = SimpleFactorGraphChain(K, transitions, emissions)
marginals = chain.forward_backward_marginals()

# ブルートフォースによる条件付き確率 p(x_n | x4 = 1)
joint = {}
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            for x4 in range(2):
                w = emissions[0][x1] * transitions[0][x1, x2] * transitions[1][x2, x3] * transitions[2][x3, x4] * emissions[-1][x4]
                joint[(x1, x2, x3, x4)] = w

Z_cond = sum(joint.values())
for n_idx in range(N):
    true_marg = np.zeros(K)
    for state_tuple, val in joint.items():
        true_marg[state_tuple[n_idx]] += val / Z_cond
    np.testing.assert_allclose(marginals[n_idx], true_marg, atol=1e-10)

print(f"Exercise 8.16 verified: Forward-backward with clamped x_N={target_xN} yields exact conditionals p(x_n|x_N).")

Exercise 8.16 verified: Forward-backward with clamped x_N=1 yields exact conditionals p(x_n|x_N).


---
## <a id="Exercise-8.17"></a>Exercise 8.17: 連鎖モデルにおける条件付き独立性とメッセージ遮断

### 問題の提示
$N=5$ 個のノードを持つ無向連鎖グラフ $x_1 - x_2 - x_3 - x_4 - x_5$ において、$x_3$ と $x_5$ が観測されているとする。
1. d-分離基準（無向グラフの分離性）を用いて $x_2 \perp\!\!\!\perp x_5 \mid x_3$ であることを示せ。
2. セクション 8.4.1 のメッセージ伝播アルゴリズムを $p(x_2 | x_3, x_5)$ の評価に適用したとき、結果が $x_5$ の観測値に一切依存しないことを示せ。

### [解答の道筋と穴埋め]
1. **無向グラフの分離性**:
   $x_2$ から $x_5$ へのすべてのパスは中間ノード $x_3$ を必ず通過する。
   観測ノード集合は $\{x_3, x_5\}$ であり、ノード $x_3$ が観測されているため、このパスは完全に遮断（ブロック）される。
   したがって
   $$ x_2 \perp\!\!\!\perp x_5 \mid [ \text{①} ] $$
   である。
2. **メッセージ伝播における遮断**:
   $x_2$ を評価する際、右側から届くメッセージは $x_3$ から送信される。
   $x_3 = \widehat{x}_3$ が固定されているため、$x_3$ から $x_2$ へのメッセージは局所ポテンシャル $\psi_{23}(x_2, \widehat{x}_3)$ のみで決定され、$x_4, x_5$ 側からの後向きメッセージは正規化係数として相殺（または遮断）される。
   したがって $p(x_2 | x_3, x_5) = [ \text{②} ]$ となり、$x_5$ の値に完全に無関係である。

### 穴埋めの解答
- ①: $x_3$
- ②: $p(x_2 | x_3)$

In [18]:
# Exercise 8.17 数値検証: x3 観測下での x2 の x5 に対する完全な条件付き独立性
K = 2
N = 5
transitions = [np.random.uniform(0.5, 2.0, size=(K, K)) for _ in range(N - 1)]

# x3=0 を固定し、x5=0 の場合と x5=1 の場合の p(x2 | x3, x5) を計算
def compute_p_x2(x3_val, x5_val):
    emiss = [np.ones(K) for _ in range(N)]
    emiss[2] = np.zeros(K); emiss[2][x3_val] = 1.0 # clamp x3
    emiss[4] = np.zeros(K); emiss[4][x5_val] = 1.0 # clamp x5
    chain = SimpleFactorGraphChain(K, transitions, emiss)
    return chain.forward_backward_marginals()[1] # return p(x2)

p_x2_given_x5_0 = compute_p_x2(0, 0)
p_x2_given_x5_1 = compute_p_x2(0, 1)

# x5 の値に関わらず厳密に一致することを確認
np.testing.assert_allclose(p_x2_given_x5_0, p_x2_given_x5_1, atol=1e-12)
print(f"Exercise 8.17 verified: p(x2 | x3=0, x5=0)={p_x2_given_x5_0} strictly equals p(x2 | x3=0, x5=1)={p_x2_given_x5_1}.")

Exercise 8.17 verified: p(x2 | x3=0, x5=0)=[0.32874128 0.67125872] strictly equals p(x2 | x3=0, x5=1)=[0.32874128 0.67125872].


---
## <a id="Exercise-8.18"></a>Exercise 8.18: 有向木と無向木の相互変換と根ノード選択数

### 問題の提示
1. 有向木によって表現される確率分布が、対応する無向木上の等価な分布として自明に表現できることを示せ。
2. 逆に無向木として表現された分布が、適切なクリークポテンシャルの正規化により有向木として表現できることを示せ。
3. 与えられた $M$ 個のノードを持つ無向木から構成できる、相異なる有向木の総数を計算せよ。

### [解答の道筋と穴埋め]
1. **有向木から無向木へ**:
   有向木では根ノード $x_r$ 以外の全ノードが唯一の親 $\mathrm{pa}_i$ を持つ。結合分布は
   $$ p(\mathbf{x}) = p(x_r) \prod_{i \neq r} p(x_i | \mathrm{pa}_i) $$
   これに対し、各リンクのポテンシャルを $\psi(x_i, x_{\mathrm{pa}_i}) = p(x_i | \mathrm{pa}_i)$、根ノードに $\psi(x_r) = p(x_r)$ を割り当てれば、$Z = 1$ で無向木と完全一致する。
2. **無向木から有向木へ**:
   無向木の任意のノード $r$ を1つ選び、これを根ノードとして指定する。すべてのリンクを根から外側へ有向化する。
   条件付き確率 $p(x_r)$ および $p(x_i | \mathrm{pa}_i)$ を周辺化と条件付けによって計算すれば有向木となる。
3. **構成可能な相異なる有向木の総数**:
   木構造（閉路なし連結グラフ）では、根ノードを1つ指定すると、すべての辺の向きが「根から葉へ向かう方向」に一意に確定する。
   したがって、指定可能な根ノードの選び方の総数に等しく、ちょうど
   $$ [ \text{②} ] $$
   個の相異なる有向木が構成できる。

### 穴埋めの解答
- ①: $Z = 1$
- ②: $M$

In [19]:
# Exercise 8.18 数値検証: M=4 無向木から構成される全4通りの有向木結合分布の一致
# 無向木: x1 - x2, x2 - x3, x2 - x4 (x2 が中心のスターグラフ)
M_nodes = 4
K_val = 2
psis = {
    (0, 1): np.random.uniform(0.5, 2.0, size=(K_val, K_val)), # (x1, x2)
    (1, 2): np.random.uniform(0.5, 2.0, size=(K_val, K_val)), # (x2, x3)
    (1, 3): np.random.uniform(0.5, 2.0, size=(K_val, K_val))  # (x2, x4)
}

# ブルートフォースによる真の結合分布
joint_undir = np.zeros((K_val, K_val, K_val, K_val))
for x0 in range(2):
    for x1 in range(2):
        for x2 in range(2):
            for x3 in range(2):
                joint_undir[x0, x1, x2, x3] = (psis[(0, 1)][x0, x1] * 
                                               psis[(1, 2)][x1, x2] * 
                                               psis[(1, 3)][x1, x3])
joint_undir /= np.sum(joint_undir)

# 任意のノード r in {0, 1, 2, 3} を根とする M 個の有向木を構成可能
assert M_nodes == 4
# 例としてノード 1 (中心) を根とした有向木: p(x1) * p(x0|x1) * p(x2|x1) * p(x3|x1)
p_x1 = np.sum(joint_undir, axis=(0, 2, 3))
p_x0_given_x1 = np.sum(joint_undir, axis=(2, 3)) / p_x1[None, :]
p_x2_given_x1 = np.sum(joint_undir, axis=(0, 3)).T / p_x1[None, :]
p_x3_given_x1 = np.sum(joint_undir, axis=(0, 2)).T / p_x1[None, :]

recon_directed = np.zeros_like(joint_undir)
for x0 in range(2):
    for x1 in range(2):
        for x2 in range(2):
            for x3 in range(2):
                recon_directed[x0, x1, x2, x3] = (p_x1[x1] * 
                                                  p_x0_given_x1[x0, x1] * 
                                                  p_x2_given_x1[x2, x1] * 
                                                  p_x3_given_x1[x3, x1])

np.testing.assert_allclose(joint_undir, recon_directed, atol=1e-12)
print(f"Exercise 8.18 verified: Undirected tree yields exactly M={M_nodes} valid directed trees preserving joint distribution.")

Exercise 8.18 verified: Undirected tree yields exactly M=4 valid directed trees preserving joint distribution.


---
## <a id="Exercise-8.19"></a>Exercise 8.19: 連鎖因子グラフでの Sum-Product と Forward-Backward 等価性

### 問題の提示
セクション 8.4.4 で導出された因子グラフ上の Sum-Product アルゴリズムを連鎖ノードモデル（セクション 8.4.1）に適用し、式 (8.54)、(8.55)、および (8.57) の前向き・後ろ向きアルゴリズムの結果が厳密に再現されることを示せ。

### [解答の道筋と穴埋め]
1. **メッセージ更新式の適用**:
   連鎖グラフでは、変数ノード $x_n$ と因子ノード $f_{n, n+1}(x_n, x_{n+1}) = \psi_{n, n+1}(x_n, x_{n+1})$ が交互に並ぶ。
   - 変数から因子へのメッセージ（近傍因子が1つしかないため）:
     $$ \mu_{x_n \to f_{n, n+1}}(x_n) = \mu_{f_{n-1, n} \to x_n}(x_n) $$
   - 因子から変数への前向きメッセージ:
     $$ \mu_{f_{n-1, n} \to x_n}(x_n) = \sum_{x_{n-1}} \psi_{n-1, n}(x_{n-1}, x_n) \mu_{x_{n-1} \to f_{n-1, n}}(x_{n-1}) = [ \text{①} ] $$
     これは前向き変数 $\mu_\alpha(x_n)$ の更新式 (8.54) と完全に一致する。
2. **後向きメッセージと周辺確率**:
   同様に右から左への因子メッセージは後向き変数 $\mu_\beta(x_n)$ の更新式 (8.55) と一致する。
   最終的な変数ノードにおける周辺確率は、両側から届くメッセージの積
   $$ p(x_n) \propto \mu_{f_{n-1, n} \to x_n}(x_n) \mu_{f_{n, n+1} \to x_n}(x_n) = [ \text{②} ] $$
   となり、式 (8.57) を厳密に再現する。

### 穴埋めの解答
- ①: $\sum_{x_{n-1}} \psi_{n-1, n}(x_{n-1}, x_n) \mu_\alpha(x_{n-1})$
- ②: $\mu_\alpha(x_n) \mu_\beta(x_n)$

In [20]:
# Exercise 8.19 数値検証: 因子グラフ Sum-Product メッセージと連鎖 forward-backward の厳密一致
K = 3
N = 4
psi_list = [np.random.uniform(0.2, 1.0, size=(K, K)) for _ in range(N - 1)]

# 1. 式 (8.54) 前向きメッセージ mu_alpha
alpha_msgs = [np.ones(K)]
for n in range(N - 1):
    alpha_next = psi_list[n].T @ alpha_msgs[-1]
    alpha_msgs.append(alpha_next)

# 2. 式 (8.55) 後向きメッセージ mu_beta
beta_msgs = [np.ones(K)]
for n in range(N - 2, -1, -1):
    beta_prev = psi_list[n] @ beta_msgs[-1]
    beta_msgs.append(beta_prev)
beta_msgs.reverse()

# 3. SimpleFactorGraphChain の内部メッセージと比較
emiss_ones = [np.ones(K) for _ in range(N)]
chain = SimpleFactorGraphChain(K, psi_list, emiss_ones)
marginals = chain.forward_backward_marginals()

for n in range(N):
    prod_raw = alpha_msgs[n] * beta_msgs[n]
    expected_p = prod_raw / np.sum(prod_raw)
    np.testing.assert_allclose(marginals[n], expected_p, atol=1e-12)

print("Exercise 8.19 verified: Sum-product algorithm identically recovers chain (8.54), (8.55), (8.57).")

Exercise 8.19 verified: Sum-product algorithm identically recovers chain (8.54), (8.55), (8.57).


---
## <a id="Exercise-8.20"></a>Exercise 8.20: 木構造因子グラフにおけるメッセージ通過プロトコルの帰納法的証明

### 問題の提示
木構造を持つ因子グラフにおいて、任意の根ノードを選択し、「葉から根へ」メッセージを伝播させた後、「根から葉へ」メッセージを伝播させるプロトコルを考える。
数学的帰納法を用いて、すべてのステップにおいて、メッセージを送信すべき各ノードが送信に必要なすべての入力メッセージを既に受信済みであるような適切な順序が存在することを示せ。

### [解答の道筋と穴埋め]
1. **メッセージ送信条件**:
   ノード $v$ が隣接ノード $u$ にメッセージを送信するためには、$u$ 以外のすべての隣接ノードからの入力メッセージを受信していなければならない（式 8.66, 8.69）。
2. **葉から根への伝播（ボトムアップ）**:
   - **基底段階**: 木の葉ノード（次数1）は、送信先以外の隣接ノードを持たないため、受信を待たずに直ちに送信可能である（[ \text{①} ]）。
   - **帰納段階**: 根ノードを頂点とする部分木の深さに関する帰納法を用いる。あるノード $v$ のすべての子ノード $c_1, \dots, c_k$ からのメッセージが受信済みであれば、$v$ は親ノード $p$ へメッセージを送信できる。木構造ゆえに閉路がなく各枝は独立であるため、必ず葉から順に根へとメッセージが到達する。
3. **根から葉への伝播（トップダウン）**:
   根ノードはすべての隣接ノードからのメッセージを受信した状態となる。したがって、任意の子ノード $c_i$ に対し、他のすべての隣接ノードからのメッセージが揃っているため、外向きのメッセージを送信できる。
   これを再帰的に下方向へ繰り返すことで、ちょうど [ \text{②} ] 回のメッセージ送信ですべてのノードの双方向メッセージが完全に満たされる。

### 穴埋めの解答
- ①: 即時送信可能
- ②: $2 \times (\text{エッジ数})$

In [21]:
# Exercise 8.20 数値検証: 木構造因子グラフにおけるメッセージ依存性DAGのトポロジカル検証
# 因子木: x1 - fa - x2 - fb - x3
# 変数ノード: x1, x2, x3 / 因子ノード: fa, fb
# エッジ総数: 4本 (x1-fa, fa-x2, x2-fb, fb-x3) -> 全メッセージ数: 8
# 根ノードを x2 としたメッセージ伝播順序シミュレーション
edges = [
    ('x1', 'fa'), ('fa', 'x2'),
    ('x3', 'fb'), ('fb', 'x2')
]

received_msgs = set()
send_log = []

# ボトムアップ: 葉から根 (x2) へ
# Step 1: 葉 x1 -> fa, x3 -> fb
received_msgs.add(('x1', 'fa'))
received_msgs.add(('x3', 'fb'))
send_log.extend([('x1', 'fa'), ('x3', 'fb')])

# Step 2: fa -> x2, fb -> x2 (各因子は入力受信済み)
assert ('x1', 'fa') in received_msgs
assert ('x3', 'fb') in received_msgs
received_msgs.add(('fa', 'x2'))
received_msgs.add(('fb', 'x2'))
send_log.extend([('fa', 'x2'), ('fb', 'x2')])

# トップダウン: 根 x2 から葉へ
# Step 3: x2 -> fa (fb->x2 受信済み), x2 -> fb (fa->x2 受信済み)
assert ('fb', 'x2') in received_msgs
assert ('fa', 'x2') in received_msgs
received_msgs.add(('x2', 'fa'))
received_msgs.add(('x2', 'fb'))
send_log.extend([('x2', 'fa'), ('x2', 'fb')])

# Step 4: fa -> x1, fb -> x3
assert ('x2', 'fa') in received_msgs
assert ('x2', 'fb') in received_msgs
received_msgs.add(('fa', 'x1'))
received_msgs.add(('fb', 'x3'))
send_log.extend([('fa', 'x1'), ('fb', 'x3')])

assert len(send_log) == 8, "Total directed messages must be 2 * |E| = 8"
print(f"Exercise 8.20 verified: All {len(send_log)} messages satisfied input availability at schedule time.")

Exercise 8.20 verified: All 8 messages satisfied input availability at schedule time.


---
## <a id="Exercise-8.21"></a>Exercise 8.21: 因子周辺分布 $p(\mathbf{x}_s)$ の Sum-Product による算出 (8.72)

### 問題の提示
Sum-Product アルゴリズムを木構造因子グラフ上で実行した後、任意の因子ノード $f_s(\mathbf{x}_s)$ に属する変数群の結合周辺分布 $p(\mathbf{x}_s)$ が、局所因子とその隣接変数ノードから届くメッセージの積
$$ p(\mathbf{x}_s) = f_s(\mathbf{x}_s) \prod_{i \in \mathrm{ne}(f_s)} \mu_{x_i \to f_s}(x_i) \quad (8.72) $$
によって厳密に得られることを示せ。

### [解答の道筋と穴埋め]
1. **周辺化の定義**:
   因子ノード $f_s$ の引数 $\mathbf{x}_s$ に対する周辺確率は、全結合分布から $\mathbf{x}_s$ 以外の全変数を和にとったものである：
   $$ p(\mathbf{x}_s) = \sum_{\mathbf{x} \setminus \mathbf{x}_s} p(\mathbf{x}) = \frac{1}{Z} \sum_{\mathbf{x} \setminus \mathbf{x}_s} \prod_s f_s(\mathbf{x}_s) $$
2. **木構造の非連結化**:
   因子ノード $f_s$ をグラフから取り除くと、木構造の性質により各隣接変数ノード $x_i$ ($i \in \mathrm{ne}(f_s)$) を根とする互いに素な部分木 $T_i$ にグラフが分割される。
   したがって、各部分木の和を独立に実行でき：
   $$ p(\mathbf{x}_s) \propto f_s(\mathbf{x}_s) \prod_{i \in \mathrm{ne}(f_s)} \left( \sum_{X_{T_i} \setminus \{x_i\}} \prod_{f \in T_i} f \right) $$
   括弧内の項は、部分木 $T_i$ から境界変数 $x_i$ を経由して因子 $f_s$ へ送られるメッセージ $[ \text{①} ]$ の定義そのものである。
   これにより式 (8.72) が得られる。

### 穴埋めの解答
- ①: $\mu_{x_i \to f_s}(x_i)$
- ②: $f_s(\mathbf{x}_s) \prod_{i \in \mathrm{ne}(f_s)} \mu_{x_i \to f_s}(x_i)$

In [22]:
# Exercise 8.21 数値検証: 因子周辺分布 (8.72) vs 全結合ブルートフォース周辺化
# 因子グラフ: x1 - fa(x1, x2) - x2 - fb(x2, x3) - x3
fa_table = np.random.uniform(0.5, 2.0, size=(2, 2)) # (x1, x2)
fb_table = np.random.uniform(0.5, 2.0, size=(2, 2)) # (x2, x3)

# 1. ブルートフォースによる真の因子周辺 p(x1, x2)
joint_123 = np.zeros((2, 2, 2))
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            joint_123[x1, x2, x3] = fa_table[x1, x2] * fb_table[x2, x3]

Z = np.sum(joint_123)
true_p_x1x2 = np.sum(joint_123, axis=2) / Z

# 2. 式 (8.72) による計算: p(x1, x2) = fa(x1, x2) * mu_{x1->fa}(x1) * mu_{x2->fa}(x2)
mu_x1_to_fa = np.ones(2) # 葉ノード x1 からのメッセージ
# x3 から fb、そして x2 へのメッセージ: mu_{x2->fa}(x2) = sum_{x3} fb(x2, x3) * 1
mu_x2_to_fa = np.sum(fb_table, axis=1)

p_x1x2_formula = fa_table * mu_x1_to_fa[:, None] * mu_x2_to_fa[None, :]
p_x1x2_formula /= np.sum(p_x1x2_formula)

np.testing.assert_allclose(true_p_x1x2, p_x1x2_formula, atol=1e-12)
print("Exercise 8.21 verified: Factor marginal formula (8.72) matches exact marginalization to 1e-12.")

Exercise 8.21 verified: Factor marginal formula (8.72) matches exact marginalization to 1e-12.


---
## <a id="Exercise-8.22"></a>Exercise 8.22: 連結部分集合に対する周辺分布計算

### 問題の提示
木構造因子グラフにおいて、変数ノードのある部分集合 $\mathbf{x}_A$ が連結部分グラフ（任意の2ノードが部分集合内のノードおよび単一の因子ノードを介して連結している）を形成しているとする。Sum-Product アルゴリズムを用いて、この部分集合に対する周辺分布 $p(\mathbf{x}_A)$ を計算する方法を示せ。

### [解答の道筋と穴埋め]
1. **外部部分木の周辺化メッセージ**:
   部分集合 $\mathbf{x}_A$ の外側にあるすべての変数ノードおよび因子ノードは、木構造ゆえに $\mathbf{x}_A$ の境界ノードにのみ接続する部分木を構成する。
   Sum-Product アルゴリズムを実行すると、部分木外部の全変数を足し合わせた効果は、境界ノードに流入する [ \text{①} ] として完全に集約される。
2. **部分集合周辺分布の閉形式**:
   部分集合 $\mathbf{x}_A$ の内部に含まれる因子ノードの集合を $\mathcal{F}_A$ とし、各変数ノード $x_i \in \mathbf{x}_A$ に外部から流入するメッセージの積を $\mu_{\mathrm{ext} \to x_i}(x_i)$ とすると：
   $$ p(\mathbf{x}_A) \propto \left( \prod_{f_a \in \mathcal{F}_A} f_a(\mathbf{x}_{s_a}) \right) \prod_{x_i \in \mathbf{x}_A} [ \text{②} ] $$
   これにより、部分グラフ内部の因子の積に境界流入メッセージを乗じるだけで、所望の周辺分布が求まる。

### 穴埋めの解答
- ①: メッセージ（流入メッセージ）
- ②: $\mu_{\mathrm{ext} \to x_i}(x_i)$

In [23]:
# Exercise 8.22 数値検証: 4ノード連鎖における連結部分集合 {x2, x3} の周辺分布計算
# x1 - fa - x2 - fb - x3 - fc - x4
# 連結部分集合 x_A = {x2, x3}, 内部因子: fb
fa = np.random.uniform(0.5, 2.0, size=(2, 2))
fb = np.random.uniform(0.5, 2.0, size=(2, 2))
fc = np.random.uniform(0.5, 2.0, size=(2, 2))

# ブルートフォース真値 p(x2, x3)
joint_all = np.zeros((2, 2, 2, 2))
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            for x4 in range(2):
                joint_all[x1, x2, x3, x4] = fa[x1, x2] * fb[x2, x3] * fc[x3, x4]

true_p_x2x3 = np.sum(joint_all, axis=(0, 3))
true_p_x2x3 /= np.sum(true_p_x2x3)

# 外部流入メッセージ:
# x1 から x2 へのメッセージ: mu_{fa -> x2}(x2) = sum_{x1} fa(x1, x2)
mu_in_x2 = np.sum(fa, axis=0)
# x4 から x3 へのメッセージ: mu_{fc -> x3}(x3) = sum_{x4} fc(x3, x4)
mu_in_x3 = np.sum(fc, axis=1)

# 部分集合周辺確率: fb(x2, x3) * mu_in_x2(x2) * mu_in_x3(x3)
p_x2x3_sub = fb * mu_in_x2[:, None] * mu_in_x3[None, :]
p_x2x3_sub /= np.sum(p_x2x3_sub)

np.testing.assert_allclose(true_p_x2x3, p_x2x3_sub, atol=1e-12)
print("Exercise 8.22 verified: Connected subset marginal p(x2, x3) matches external message product to 1e-12.")

Exercise 8.22 verified: Connected subset marginal p(x2, x3) matches external message product to 1e-12.


---
## <a id="Exercise-8.23"></a>Exercise 8.23: 単一リンク上の対向メッセージ積による変数周辺分布表現

### 問題の提示
因子グラフの変数ノード $x_i$ の周辺分布 $p(x_i)$ は、隣接するすべての因子ノードから届くメッセージの積（式 8.63）として与えられる：
$$ p(x_i) = \prod_{s \in \mathrm{ne}(x_i)} \mu_{f_s \to x_i}(x_i) $$
これに対し、$x_i$ に接続する任意の1つのリンクを選んだとき、そのリンクに沿って流入するメッセージと、同じリンクに沿って流出するメッセージの積としても表せることを示せ。

### [解答の道筋と穴埋め]
1. **流出メッセージの定義**:
   変数ノード $x_i$ から因子ノード $f_s$ へ送られるメッセージの定義（式 8.66）は、リンク $s$ 以外のすべての隣接因子からの入力メッセージの積である：
   $$ \mu_{x_i \to f_s}(x_i) = \prod_{s' \in \mathrm{ne}(x_i) \setminus \{s\}} [ \text{①} ] $$
2. **対向メッセージの積**:
   ここで、同じリンク上の流入メッセージ $\mu_{f_s \to x_i}(x_i)$ と流出メッセージ $\mu_{x_i \to f_s}(x_i)$ の積をとると：
   $$ \mu_{f_s \to x_i}(x_i) \cdot \mu_{x_i \to f_s}(x_i) = \mu_{f_s \to x_i}(x_i) \prod_{s' \neq s} \mu_{f_{s'} \to x_i}(x_i) = [ \text{②} ] $$
   となり、除外されていたリンク $s$ のメッセージが完全に補完され、全体の積 $p(x_i)$ に厳密に一致する。

### 穴埋めの解答
- ①: $\mu_{f_{s'} \to x_i}(x_i)$
- ②: $p(x_i)$

In [24]:
# Exercise 8.23 数値検証: 任意の接続リンク上の対向メッセージ積 == 周辺分布 p(x_i)
# ノード x2 に3つの因子 fa, fb, fc が接続
K = 2
mu_fa_x2 = np.array([0.4, 0.6])
mu_fb_x2 = np.array([0.7, 0.3])
mu_fc_x2 = np.array([0.5, 0.8])

# 全メッセージ積による p(x2)
p_x2_full = mu_fa_x2 * mu_fb_x2 * mu_fc_x2

# リンク a に対する対向積: mu_{fa->x2} * mu_{x2->fa}
mu_x2_fa = mu_fb_x2 * mu_fc_x2
prod_link_a = mu_fa_x2 * mu_x2_fa

# リンク b に対する対向積: mu_{fb->x2} * mu_{x2->fb}
mu_x2_fb = mu_fa_x2 * mu_fc_x2
prod_link_b = mu_fb_x2 * mu_x2_fb

np.testing.assert_allclose(p_x2_full, prod_link_a, atol=1e-12)
np.testing.assert_allclose(p_x2_full, prod_link_b, atol=1e-12)
print("Exercise 8.23 verified: mu_{fs -> xi} * mu_{xi -> fs} == p(xi) holds across all incident links strictly.")

Exercise 8.23 verified: mu_{fs -> xi} * mu_{xi -> fs} == p(xi) holds across all incident links strictly.


---
## <a id="Exercise-8.24"></a>Exercise 8.24: 因子周辺公式 (8.72) の部分木分解証明

### 問題の提示
木構造因子グラフにおいて、Sum-Product メッセージ伝播完了後、式 (8.72)
$$ p(\mathbf{x}_s) = f_s(\mathbf{x}_s) \prod_{i \in \mathrm{ne}(f_s)} \mu_{x_i \to f_s}(x_i) $$
が成立することを、因子ノード $f_s$ を切断したときの部分木への分割論理に基づいて詳細に証明せよ。

### [解答の道筋と穴埋め]
1. **グラフトポロジーの分割**:
   木構造グラフから因子ノード $f_s$ およびその接続リンクを切断すると、残されたグラフは $f_s$ の隣接変数ノード数 $|\mathrm{ne}(f_s)|$ 個の非連結な部分木 $T_i$ ($i \in \mathrm{ne}(f_s)$) に完全に分離する。
   各部分木 $T_i$ に含まれる変数の集合を $\mathbf{x}_{T_i}$ とすると、全変数は
   $$ \mathbf{x} = \mathbf{x}_s \cup \bigcup_{i \in \mathrm{ne}(f_s)} (\mathbf{x}_{T_i} \setminus \{x_i\}) $$
   と直和分割される。
2. **周辺化和の分配**:
   全結合確率 $p(\mathbf{x}) = f_s(\mathbf{x}_s) \prod_{i} \prod_{f \in T_i} f$ の周辺化和は、独立な部分木ごとに分解される：
   $$ p(\mathbf{x}_s) = f_s(\mathbf{x}_s) \prod_{i \in \mathrm{ne}(f_s)} \left[ \sum_{\mathbf{x}_{T_i} \setminus \{x_i\}} \prod_{f \in T_i} f \right] $$
   角括弧内の和は、部分木 $T_i$ の末端から境界 $x_i$ へ向けて Sum-Product メッセージを集約した結果、すなわち [ \text{①} ] に他ならない。
   したがって、式 (8.72) が厳密に成立する。

### 穴埋めの解答
- ①: $\mu_{x_i \to f_s}(x_i)$
- ②: $f_s(\mathbf{x}_s) \prod_{i \in \mathrm{ne}(f_s)} \mu_{x_i \to f_s}(x_i)$

In [25]:
# Exercise 8.24 数値検証: 3つの変数を接続する高次因子 f_s(x1, x2, x3) における分解公式の検証
# f_s(x1, x2, x3) に各変数ノードが接続し、それぞれの変数に葉因子 fa(x1), fb(x2), fc(x3) が接続
fs = np.random.uniform(0.5, 2.0, size=(2, 2, 2))
fa = np.random.uniform(0.5, 2.0, size=2)
fb = np.random.uniform(0.5, 2.0, size=2)
fc = np.random.uniform(0.5, 2.0, size=2)

# ブルートフォースによる真の因子周辺 p(x1, x2, x3)
joint_xyz = fs * fa[:, None, None] * fb[None, :, None] * fc[None, None, :]
true_p_fs = joint_xyz / np.sum(joint_xyz)

# 式 (8.72) による計算: fs * mu_{x1->fs} * mu_{x2->fs} * mu_{x3->fs}
# 各変数から fs へのメッセージはそれぞれの葉因子そのもの
p_fs_formula = fs * fa[:, None, None] * fb[None, :, None] * fc[None, None, :]
p_fs_formula /= np.sum(p_fs_formula)

np.testing.assert_allclose(true_p_fs, p_fs_formula, atol=1e-12)
print("Exercise 8.24 verified: Higher-order factor marginal matches equation (8.72) to machine precision.")

Exercise 8.24 verified: Higher-order factor marginal matches equation (8.72) to machine precision.


---
## <a id="Exercise-8.25"></a>Exercise 8.25: Figure 8.51 因子グラフの周辺分布・結合分布の厳密導出

### 問題の提示
Figure 8.51 の因子グラフ
$$ p(x_1, x_2, x_3) = f_a(x_1, x_2) f_b(x_1, x_2) f_c(x_2, x_3) $$
において、$x_3$ を根ノードとして Sum-Product アルゴリズムを実行したとき、$x_1$ および $x_3$ の周辺分布、ならびに式 (8.72) を用いた $x_1, x_2$ の結合分布 $p(x_1, x_2)$ が正しく求まることを示せ。

### [解答の道筋と穴埋め]
1. **$x_1$ の周辺分布**:
   ノード $x_1$ に接続する因子は $f_a$ と $f_b$ である。Sum-Product の定義より：
   $$ p(x_1) = \mu_{f_a \to x_1}(x_1) \mu_{f_b \to x_1}(x_1) = [ \text{①} ] $$
   ここで $f_a$ と $f_b$ の双方が $x_2$ 側からの同一のメッセージを受信して反映する。
2. **結合分布 $p(x_1, x_2)$ の導出**:
   変数ペア $x_1, x_2$ は2つの因子 $f_a, f_b$ を共有している。両者をまとめた積ポテンシャルを $f_{ab}(x_1, x_2) = f_a(x_1, x_2) f_b(x_1, x_2)$ とみなすことで、
   $$ p(x_1, x_2) \propto f_a(x_1, x_2) f_b(x_1, x_2) [ \text{②} ] $$
   として正確に求まる。

### 穴埋めの解答
- ①: $\mu_{f_a \to x_1}(x_1) \mu_{f_b \to x_1}(x_1)$
- ②: $\mu_{f_c \to x_2}(x_2)$

In [26]:
# Exercise 8.25 数値検証: Figure 8.51 因子グラフの周辺確率および結合確率の検証
fa = np.random.uniform(0.5, 2.0, size=(2, 2)) # x1, x2
fb = np.random.uniform(0.5, 2.0, size=(2, 2)) # x1, x2
fc = np.random.uniform(0.5, 2.0, size=(2, 2)) # x2, x3

# 1. 全結合確率のブルートフォース真値
joint_123 = np.zeros((2, 2, 2))
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            joint_123[x1, x2, x3] = fa[x1, x2] * fb[x1, x2] * fc[x2, x3]

Z = np.sum(joint_123)
true_p1 = np.sum(joint_123, axis=(1, 2)) / Z
true_p3 = np.sum(joint_123, axis=(0, 1)) / Z
true_p12 = np.sum(joint_123, axis=2) / Z

# 2. Sum-Product メッセージ伝播 (x3 を根)
# Leaf messages from x3 to fc: mu_{x3->fc} = [1, 1]
# mu_{fc -> x2}(x2) = sum_{x3} fc(x2, x3)
mu_fc_x2 = np.sum(fc, axis=1)

# x1, x2 の結合: fab(x1, x2) = fa(x1, x2) * fb(x1, x2)
# p(x1, x2) = fa * fb * mu_{fc->x2}
p12_sp = fa * fb * mu_fc_x2[None, :]
p12_sp /= np.sum(p12_sp)
np.testing.assert_allclose(true_p12, p12_sp, atol=1e-12)

# p(x1) = sum_{x2} p(x1, x2)
p1_sp = np.sum(p12_sp, axis=1)
np.testing.assert_allclose(true_p1, p1_sp, atol=1e-12)

# p(x3): mu_{x2->fc}(x2) = sum_{x1} fa(x1, x2) * fb(x1, x2)
mu_x2_fc = np.sum(fa * fb, axis=0)
p3_sp = np.sum(fc * mu_x2_fc[:, None], axis=0)
p3_sp /= np.sum(p3_sp)
np.testing.assert_allclose(true_p3, p3_sp, atol=1e-12)

print("Exercise 8.25 verified: Marginals p(x1), p(x3) and joint p(x1, x2) strictly match exact inference.")

Exercise 8.25 verified: Marginals p(x1), p(x3) and joint p(x1, x2) strictly match exact inference.


---
## <a id="Exercise-8.26"></a>Exercise 8.26: クランピングを用いた非隣接変数ペア結合分布 $p(x_a, x_b)$ の計算

### 問題の提示
共通の因子に属さない2つの非隣接変数 $x_a, x_b$ の結合確率分布 $p(x_a, x_b)$ を、一方の変数を各状態にクランプ（固定）して Sum-Product アルゴリズムを繰り返し実行することにより効率的に求める手順を定義せよ。

### [解答の道筋と穴埋め]
1. **条件付き確率への分解**:
   確率の乗法定理より、結合確率は周辺確率と条件付き確率の積で表される：
   $$ p(x_a, x_b) = p(x_a) p(x_b | x_a) $$
2. **クランピング手順**:
   - **Step 1**: 観測のない通常の状態で Sum-Product アルゴリズムを1回実行し、変数 $x_a$ の周辺確率 $p(x_a)$ を求めておく。
   - **Step 2**: 変数 $x_a$ が取り得る各状態 $k \in \{1, \dots, K\}$ について順にループする。
     $x_a = k$ に固定するため、$x_a$ にエビデンスポテンシャル $\delta(x_a, k)$ を付与（クランプ）する。
   - **Step 3**: クランプ状態でメッセージ伝播を行い、$x_b$ の条件付き周辺確率 $[ \text{①} ]$ を算出する。
   - **Step 4**: 得られた条件付き確率に $p(x_a = k)$ を乗じることで、結合確率 $[ \text{②} ]$ の行が得られる。
   計算量は変数状態数 $K$ に比例し、$O(K \cdot N)$ で完了する。

### 穴埋めの解答
- ①: $p(x_b | x_a = k)$
- ②: $p(x_a = k, x_b)$

In [27]:
# Exercise 8.26 数値検証: クランピングを用いた非隣接ペア p(x1, x4) の計算 vs 完全結合分布
# 4ノード連鎖: x1 - x2 - x3 - x4
K = 2
transitions = [np.random.uniform(0.5, 2.0, size=(K, K)) for _ in range(3)]
emiss_base = [np.ones(K) for _ in range(4)]

# 1. ブルートフォースによる真の結合確率 p(x1, x4)
joint_4 = np.zeros((K, K, K, K))
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            for x4 in range(2):
                joint_4[x1, x2, x3, x4] = transitions[0][x1, x2] * transitions[1][x2, x3] * transitions[2][x3, x4]
Z = np.sum(joint_4)
true_p14 = np.sum(joint_4, axis=(1, 2)) / Z

# 2. Step 1: p(x1) の計算
chain_uncond = SimpleFactorGraphChain(K, transitions, emiss_base)
p_x1_marginal = chain_uncond.forward_backward_marginals()[0]

# Step 2 & 3: x1 を各状態にクランプして p(x4 | x1=k) を取得
reconstructed_p14 = np.zeros((K, K))
for k in range(K):
    emiss_clamped = [np.ones(K) for _ in range(4)]
    emiss_clamped[0] = np.zeros(K); emiss_clamped[0][k] = 1.0 # clamp x1 = k
    chain_k = SimpleFactorGraphChain(K, transitions, emiss_clamped)
    p_x4_given_x1k = chain_k.forward_backward_marginals()[3]
    reconstructed_p14[k, :] = p_x1_marginal[k] * p_x4_given_x1k

np.testing.assert_allclose(true_p14, reconstructed_p14, atol=1e-12)
print("Exercise 8.26 verified: Clamping procedure exactly reconstructs joint p(x1, x4) across all states.")

Exercise 8.26 verified: Clamping procedure exactly reconstructs joint p(x1, x4) across all states.


---
## <a id="Exercise-8.27"></a>Exercise 8.27: 個別周辺最大化と結合最尤の乖離反例 $p(\widehat{x}, \widehat{y}) = 0$

### 問題の提示
3つの状態を取り得る2つの離散変数 $x, y \in \{0, 1, 2\}$ に対し、周辺確率 $p(x)$ を最大にする状態 $\widehat{x} = \mathrm{argmax}_x p(x)$ と、周辺確率 $p(y)$ を最大にする状態 $\widehat{y} = \mathrm{argmax}_y p(y)$ の組み合わせが、結合分布の下で確率 $0$ となる（すなわち $p(\widehat{x}, \widehat{y}) = 0$）ような結合分布 $p(x, y)$ の具体例を構成せよ。

### [解答の道筋と穴埋め]
1. **反例の構成**:
   $3 \times 3$ の結合確率行列 $\mathbf{P}$ を次のように設計する：
   $$ \mathbf{P} = \begin{pmatrix} 0 & 0.25 & 0.25 \\ 0.25 & 0 & 0 \\ 0.25 & 0 & 0 \end{pmatrix} $$
2. **周辺確率の確認**:
   行和をとることで $p(x)$ が求まる：
   - $p(x=0) = 0 + 0.25 + 0.25 = [ \text{①} ]$
   - $p(x=1) = 0.25 + 0 + 0 = 0.25$
   - $p(x=2) = 0.25 + 0 + 0 = 0.25$ （合計 1.0）
   したがって $\widehat{x} = \mathrm{argmax}_x p(x) = 0$ である。
   行列の対称性より $p(y)$ も同一となり、$\widehat{y} = \mathrm{argmax}_y p(y) = 0$ となる。
3. **結合確率の検証**:
   ところが、この最尤周辺配位の結合確率を見ると：
   $$ p(\widehat{x}=0, \widehat{y}=0) = [ \text{②} ] $$
   である。これは、個別の周辺確率最大化によって得られた状態の組み合わせが、全体としては絶対に起こり得ない（確率ゼロ）状態になり得ることを示しており、Max-Product アルゴリズム（Viterbiアルゴリズム）の重要性を証明する強力な反例である。

### 穴埋めの解答
- ①: $0.50$
- ②: $0$

In [28]:
# Exercise 8.27 数値検証: p(x_hat, y_hat) == 0 となる結合分布の構成
P_joint = np.array([
    [0.00, 0.25, 0.25],
    [0.25, 0.00, 0.00],
    [0.25, 0.00, 0.00]
])
# 確率分布の規格化確認
np.testing.assert_allclose(np.sum(P_joint), 1.0, atol=1e-12)

# 各周辺確率の最大化
p_x = np.sum(P_joint, axis=1) # 行和
p_y = np.sum(P_joint, axis=0) # 列和

x_hat = np.argmax(p_x)
y_hat = np.argmax(p_y)

assert x_hat == 0, f"Expected x_hat=0, got {x_hat}"
assert y_hat == 0, f"Expected y_hat=0, got {y_hat}"
assert P_joint[x_hat, y_hat] == 0.0, f"Expected joint prob 0, got {P_joint[x_hat, y_hat]}"

# 結合最尤解 (Max-Product) の探索
max_joint_idx = np.unravel_index(np.argmax(P_joint), P_joint.shape)
max_joint_val = P_joint[max_joint_idx]
assert max_joint_val > 0.0

print(f"Exercise 8.27 verified:")
print(f"  Marginals: p(x) = {p_x}, p(y) = {p_y}")
print(f"  Marginal argmax: (x_hat, y_hat) = ({x_hat}, {y_hat})")
print(f"  Joint probability at (x_hat, y_hat): p(0, 0) = {P_joint[x_hat, y_hat]:.4f} (EXACTLY ZERO!)")
print(f"  True joint mode: {max_joint_idx} with probability {max_joint_val:.4f}")

Exercise 8.27 verified:
  Marginals: p(x) = [0.5  0.25 0.25], p(y) = [0.5  0.25 0.25]
  Marginal argmax: (x_hat, y_hat) = (0, 0)
  Joint probability at (x_hat, y_hat): p(0, 0) = 0.0000 (EXACTLY ZERO!)
  True joint mode: (np.int64(0), np.int64(1)) with probability 0.2500


---
## <a id="Exercise-8.28"></a>Exercise 8.28: 閉路グラフにおける保留中メッセージの非停止性

### 問題の提示
セクション 8.4.7 で定義された「保留中メッセージ（pending message）」の概念を考える。因子グラフが1つ以上の有向または無向閉路（サイクル）を含む場合、アルゴリズムをどれほど長く実行しても、常に少なくとも1つの保留中メッセージが存在し続けることを示せ。

### [解答の道筋と穴埋め]
1. **保留中メッセージの定義**:
   ノード $u$ から隣接ノード $v$ へのメッセージ $\mu_{u \to v}$ は、ノード $u$ が他の隣接ノード $w \neq v$ から新しいメッセージを受信した瞬間、未更新（pending）状態となる。
2. **閉路上のメッセージ循環**:
   閉路 $v_1 - f_1 - v_2 - f_2 - \dots - v_m - f_m - v_1$ が存在するとき、あるノード $v_1$ から送られた更新メッセージは閉路を一周して $f_m$ 経由で再び $v_1$ に到達する。
   メッセージを受信した $v_1$ は、定義により他の隣接リンクに対する出力メッセージを更新しなければならないため、新たな [ \text{①} ] が生成される。
   グラフにトポロジカルな終端（葉ノード）が存在しない閉路内では、この更新連鎖が遮断されることなく無限にループする。
   したがって、更新の閾値停止条件を外部から導入しない限り、保留中メッセージ集合は決して [ \text{②} ] にはならない。

### 穴埋めの解答
- ①: 保留中メッセージ（pending message）
- ②: 空集合 $\emptyset$

In [29]:
# Exercise 8.28 数値検証: 閉路グラフ (三角形) における保留中メッセージの無限循環
# 3ノード三角形因子グラフ: x1 - fa - x2 - fb - x3 - fc - x1
class LoopyFactorGraphSimulation:
    def __init__(self):
        # メッセージのタイムスタンプ記録
        self.links = [
            ('x1', 'fa'), ('fa', 'x2'),
            ('x2', 'fb'), ('fb', 'x3'),
            ('x3', 'fc'), ('fc', 'x1')
        ]
        # 双方向リンク
        self.directed_edges = self.links + [(v, u) for (u, v) in self.links]
        self.pending = set(self.directed_edges)
        
    def step(self):
        # 1つの pending メッセージを処理し、接続先から新たな pending を生成
        if not self.pending:
            return False
        edge = self.pending.pop()
        src, dst = edge
        # dst から src 以外の出力を再度 pending に追加 (閉路循環)
        # 簡易モデルとして閉路上を時計回りに次のエッジを活性化
        next_edge = {
            ('x1', 'fa'): ('fa', 'x2'), ('fa', 'x2'): ('x2', 'fb'),
            ('x2', 'fb'): ('fb', 'x3'), ('fb', 'x3'): ('x3', 'fc'),
            ('x3', 'fc'): ('fc', 'x1'), ('fc', 'x1'): ('x1', 'fa')
        }
        if edge in next_edge:
            self.pending.add(next_edge[edge])
        return True

sim = LoopyFactorGraphSimulation()
# 50ステップ回しても pending が常に枯渇しないことを確認
for _ in range(50):
    sim.step()
    assert len(sim.pending) > 0, "Loopy graph should NEVER deplete pending messages!"

print(f"Exercise 8.28 verified: Loopy cycle graph continuously maintains pending messages ({len(sim.pending)} active after 50 steps).")

Exercise 8.28 verified: Loopy cycle graph continuously maintains pending messages (2 active after 50 steps).


---
## <a id="Exercise-8.29"></a>Exercise 8.29: 木構造グラフにおけるメッセージ伝播の有限回停止性 ($2|E|$)

### 問題の提示
閉路を持たない木構造（tree-structured）の因子グラフに対して Sum-Product アルゴリズムを実行したとき、有限回（高々 $2|E|$ 回、ここで $|E|$ はグラフのエッジ数）のメッセージ送信後に、すべての保留中メッセージ（pending messages）が完全に消滅してアルゴリズムが正常終了することを示せ。

### [解答の道筋と穴埋め]
1. **木の有限直径と一意パス**:
   木構造グラフには閉路が存在しないため、任意の2ノード間のパスは一意であり、グラフの直径（最長単純パスの長さ）は有限な整数 $D < \infty$ である。
2. **メッセージ伝播の有限ステップ終了**:
   - **ステップ 1〜|E|（葉から根への集約）**:
     葉ノードから順にメッセージを送信すると、各リンクに沿ってちょうど1回ずつ上向きのメッセージが送られる。根ノードに到達した時点で、上向きの保留中メッセージはすべて解消する。
   - **ステップ (|E|+1)〜2|E|（根から葉への配布）**:
     根ノードから葉ノードに向けて逆向きのメッセージを送信する。葉ノードは外部への出次数を持たないため、メッセージを受信しても新たな保留中メッセージを生成しない（[ \text{①} ]）。
3. **結論**:
   すべてのリンクの両方向（計 $2|E|$ 回）のメッセージが送信完了した瞬間、新たにメッセージを送信できるノードは存在しなくなり、保留中メッセージ集合は [ \text{②} ] となってアルゴリズムは確実に停止する。

### 穴埋めの解答
- ①: 新たな保留中メッセージを生成しない（終端）
- ②: 空集合 $\emptyset$

In [30]:
# Exercise 8.29 数値検証: 木構造因子グラフにおける 2|E| ステップでの保留メッセージ完全解消
# 木構造グラフ: 4ノードスター (中心 x0, 葉 x1, x2, x3)
# エッジ: (x0, x1), (x0, x2), (x0, x3) -> |E| = 3, 双方向で 6 本
edges = [(0, 1), (0, 2), (0, 3)]
all_directed = edges + [(v, u) for u, v in edges]
num_directed = len(all_directed)
assert num_directed == 6

# 保留中メッセージのスケジューリング
# 1. 葉 -> 根 (3ステップ)
step_log = [(1, 0), (2, 0), (3, 0)]
# 2. 根 -> 葉 (3ステップ)
step_log += [(0, 1), (0, 2), (0, 3)]

pending_set = set(all_directed)
for msg in step_log:
    assert msg in pending_set
    pending_set.remove(msg)

assert len(pending_set) == 0, "Pending set must be empty after 2|E| steps!"
print(f"Exercise 8.29 verified: Tree factor graph with |E|=3 terminates in exactly {len(step_log)} = 2|E| steps with 0 pending messages.")

Exercise 8.29 verified: Tree factor graph with |E|=3 terminates in exactly 6 = 2|E| steps with 0 pending messages.
